In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mplsoccer import Radar, grid
from matplotlib import font_manager
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURACIÓN DE COLORES PERSONALIZABLE
# ============================================
"""
Personaliza los colores aquí según necesites:
- COLOR_JUGADOR: Color principal del jugador analizado
- COLOR_COMPARACION: Color del segundo jugador o promedio
- COLOR_FONDO: Fondo de toda la visualización
- COLOR_ANILLOS: Anillos concéntricos del radar
- COLOR_TEXTO: Color del texto (labels, títulos)
"""

COLOR_JUGADOR = '#00f2c1'        # Cyan por defecto - PERSONALIZABLE
COLOR_COMPARACION = '#ff6b35'    # Naranja por defecto - PERSONALIZABLE (antes era gris)
COLOR_FONDO = '#24282a'          # Fondo oscuro
COLOR_ANILLOS = '#3a3f42'        # Anillos internos
COLOR_BORDE_ANILLOS = '#808080'  # Borde de anillos
COLOR_TEXTO = '#ffffff'          # Texto blanco
COLOR_ENDNOTE = '#808080'        # Texto secundario

# ============================================
# 1. CARGAR Y PREPARAR DATOS
# ============================================

# Cargar datos
df = pd.read_csv(r'C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\fbref\jugadores_campo_2025_2026.csv')

# Filtrar jugadores con mínimo 300 minutos
df = df[df['minutos'] >= 300].copy()

print(f"Total jugadores con +300 min: {len(df)}")
print(f"\nCompeticiones disponibles:\n{df['competicion'].value_counts()}")
print(f"\nPosiciones:\n{df['posicion'].value_counts()}")

# ============================================
# 2. DEFINIR MÉTRICAS POR POSICIÓN
# ============================================

METRICAS_POR_POSICION = {
    'FW': {
        'metricas': [
            'goles_por90',
            'npxg_por90', 
            'tiros_a_puerta_por90',
            'xg_asistencias_por90',
            'pases_progresivos',
            'conducciones_progresivas',
            'regates_exitosos',
            'toques_area_rival',
            'pases_tercio_final'
        ],
        'labels': [
            'Goles p90',
            'npxG p90',
            'Tiros a puerta p90',
            'xA p90',
            'Pases progresivos',
            'Conducciones progresivas',
            'Regates exitosos',
            'Toques área rival',
            'Pases tercio final'
        ]
    },
    'MF': {
        'metricas': [
            'xg_asistencias_por90',
            'pases_progresivos',
            'conducciones_progresivas',
            'porc_precision_pase',
            'pases_tercio_final',
            'goles_asistencias_por90',
            'entradas_ganadas',
            'intercepciones',
            'recuperaciones'
        ],
        'labels': [
            'xA p90',
            'Pases progresivos',
            'Conducciones progresivas',
            '% Precisión pase',
            'Pases tercio final',
            'G+A p90',
            'Entradas ganadas',
            'Intercepciones',
            'Recuperaciones'
        ]
    },
    'DF': {
        'metricas': [
            'entradas_ganadas',
            'intercepciones',
            'duelos_aereos_ganados',
            'porc_duelos_aereos_ganados',
            'despejes',
            'bloqueos',
            'recuperaciones',
            'porc_precision_pase',
            'pases_progresivos'
        ],
        'labels': [
            'Entradas ganadas',
            'Intercepciones',
            'Duelos aéreos ganados',
            '% Duelos aéreos',
            'Despejes',
            'Bloqueos',
            'Recuperaciones',
            '% Precisión pase',
            'Pases progresivos'
        ]
    }
}

# ============================================
# 3. FUNCIONES AUXILIARES
# ============================================

def obtener_datos_jugador(df_liga, jugador_nombre, posicion, modo='percentiles'):
    """
    Obtiene los datos de un jugador según el modo especificado
    modo: 'percentiles' o 'valores_reales'
    """
    # Filtrar por posición
    df_posicion = df_liga[df_liga['posicion'] == posicion].copy()
    
    # Obtener datos del jugador
    jugador = df_posicion[df_posicion['jugador'] == jugador_nombre].iloc[0]
    
    # Métricas a usar
    metricas = METRICAS_POR_POSICION[posicion]['metricas']
    labels = METRICAS_POR_POSICION[posicion]['labels']
    
    valores_jugador = []
    valores_promedio = []
    percentiles = []
    
    for metrica in metricas:
        valor_jugador = jugador[metrica]
        promedio_liga = df_posicion[metrica].mean()
        
        valores_jugador.append(valor_jugador)
        valores_promedio.append(promedio_liga)
        
        # Calcular percentil
        percentil = (df_posicion[metrica] <= valor_jugador).sum() / len(df_posicion) * 100
        percentiles.append(percentil)
    
    if modo == 'percentiles':
        return percentiles, valores_promedio, labels, jugador, percentiles, valores_jugador
    else:  # valores_reales
        return valores_jugador, valores_promedio, labels, jugador, percentiles, valores_jugador


def obtener_valores_comparacion(df_liga, jugador1_nombre, jugador2_nombre, posicion, modo='percentiles'):
    """
    Obtiene valores para comparar dos jugadores
    modo: 'percentiles' o 'valores_reales'
    """
    # Filtrar por posición
    df_posicion = df_liga[df_liga['posicion'] == posicion].copy()
    
    # Obtener datos de ambos jugadores
    jugador1 = df_posicion[df_posicion['jugador'] == jugador1_nombre].iloc[0]
    jugador2 = df_posicion[df_posicion['jugador'] == jugador2_nombre].iloc[0]
    
    # Métricas a usar
    metricas = METRICAS_POR_POSICION[posicion]['metricas']
    labels = METRICAS_POR_POSICION[posicion]['labels']
    
    valores1 = []
    valores2 = []
    
    for metrica in metricas:
        valor_j1 = jugador1[metrica]
        valor_j2 = jugador2[metrica]
        
        if modo == 'percentiles':
            # Calcular percentiles
            percentil_j1 = (df_posicion[metrica] <= valor_j1).sum() / len(df_posicion) * 100
            percentil_j2 = (df_posicion[metrica] <= valor_j2).sum() / len(df_posicion) * 100
            valores1.append(percentil_j1)
            valores2.append(percentil_j2)
        else:  # valores_reales
            valores1.append(valor_j1)
            valores2.append(valor_j2)
    
    return valores1, valores2, labels, jugador1, jugador2


# ============================================
# 4. FUNCIÓN RADAR CHART - INDIVIDUAL
# ============================================

def crear_radar_individual(jugador_nombre, competicion='La Liga', modo='percentiles',
                          guardar=False, filename='radar_individual.png',
                          color_jugador=None, color_promedio=None):
    """
    Crea un radar chart de un jugador
    
    Parámetros:
    - jugador_nombre: Nombre del jugador
    - competicion: 'La Liga', 'Premier League', 'Serie A', 'Bundesliga', 'Ligue 1', o 'Top 5 Ligas'
    - modo: 'percentiles' o 'valores_reales'
    - color_jugador: Color personalizado para el jugador (opcional)
    - color_promedio: Color personalizado para el promedio (opcional, solo en valores_reales)
    """
    # Usar colores personalizados o los por defecto
    color_jug = color_jugador if color_jugador else COLOR_JUGADOR
    color_prom = color_promedio if color_promedio else COLOR_COMPARACION
    
    # Filtrar por competición
    if competicion == 'Top 5 Ligas':
        df_liga = df.copy()
    else:
        df_liga = df[df['competicion'] == competicion].copy()
    
    # Verificar que el jugador existe
    if jugador_nombre not in df_liga['jugador'].values:
        print(f"❌ Jugador '{jugador_nombre}' no encontrado en {competicion}")
        return
    
    # Obtener posición del jugador
    posicion = df_liga[df_liga['jugador'] == jugador_nombre]['posicion'].values[0]
    
    # Obtener datos
    valores_jugador, valores_promedio, labels, jugador, percentiles, valores_reales = obtener_datos_jugador(
        df_liga, jugador_nombre, posicion, modo
    )
    
    # Configurar radar según el modo
    if modo == 'percentiles':
        low = [0] * len(labels)
        high = [100] * len(labels)
    else:  # valores_reales
        # CORRECCIÓN: Limitar valores de porcentajes a 100 y calcular rangos dinámicos
        low = [0] * len(labels)
        high = []
        
        # Clipear valores de porcentajes a 100
        for i, label in enumerate(labels):
            if '%' in label:
                valores_jugador[i] = min(valores_jugador[i], 100)
                valores_promedio[i] = min(valores_promedio[i], 100)
        
        # Calcular rangos
        for i, (label, val_jug, val_prom) in enumerate(zip(labels, valores_jugador, valores_promedio)):
            max_val = max(val_jug, val_prom)
            if '%' in label:
                high.append(100)
            else:
                high.append(max_val * 1.2)
    
    radar = Radar(labels, low, high,
                  num_rings=4,
                  ring_width=1,
                  center_circle_radius=1)
    
    # Crear figura
    fig, axs = grid(figheight=14, grid_height=0.915, title_height=0.06, endnote_height=0.025,
                    title_space=0, endnote_space=0, grid_key='radar', axis=False)
    
    # Configurar fondo COMPLETO
    fig.patch.set_facecolor(COLOR_FONDO)
    for ax_name in axs:
        axs[ax_name].patch.set_facecolor(COLOR_FONDO)
    
    # Plot radar
    radar.setup_axis(ax=axs['radar'], facecolor=COLOR_FONDO)
    axs['radar'].set_facecolor(COLOR_FONDO)
    
    rings_inner = radar.draw_circles(ax=axs['radar'], facecolor=COLOR_ANILLOS, 
                                     edgecolor=COLOR_BORDE_ANILLOS, linewidth=1.5)
    
    if modo == 'percentiles':
        # Solo mostrar el jugador en percentiles
        valores_base = [0] * len(valores_jugador)
        radar_output = radar.draw_radar_compare(valores_base, valores_jugador, ax=axs['radar'],
                                                kwargs_radar={'facecolor': COLOR_ANILLOS, 'alpha': 0.0},
                                                kwargs_compare={'facecolor': color_jug, 'alpha': 0.6, 
                                                              'edgecolor': color_jug, 'linewidth': 2})
        radar_poly, radar_poly2, vertices1, vertices2 = radar_output
        vertices_jugador = vertices2
        
        # Solo scatter del jugador
        axs['radar'].scatter(vertices_jugador[:, 0], vertices_jugador[:, 1],
                           c=color_jug, edgecolors=COLOR_TEXTO, marker='o', s=150, zorder=2)
        
    else:  # valores_reales
        # Mostrar jugador vs promedio
        radar_output = radar.draw_radar_compare(valores_promedio, valores_jugador, ax=axs['radar'],
                                                kwargs_radar={'facecolor': color_prom, 'alpha': 0.4, 
                                                            'edgecolor': color_prom, 'linewidth': 2},
                                                kwargs_compare={'facecolor': color_jug, 'alpha': 0.6, 
                                                              'edgecolor': color_jug, 'linewidth': 2})
        radar_poly_prom, radar_poly_jug, vertices_prom, vertices_jugador = radar_output
        
        # Scatter de ambos
        axs['radar'].scatter(vertices_prom[:, 0], vertices_prom[:, 1],
                           c=color_prom, edgecolors=COLOR_TEXTO, marker='o', s=150, zorder=2)
        axs['radar'].scatter(vertices_jugador[:, 0], vertices_jugador[:, 1],
                           c=color_jug, edgecolors=COLOR_TEXTO, marker='o', s=150, zorder=2)
    
    # Range labels
    range_labels = radar.draw_range_labels(ax=axs['radar'], fontsize=20, color=COLOR_TEXTO)
    
    # Param labels
    param_labels = radar.draw_param_labels(ax=axs['radar'], fontsize=22, color=COLOR_TEXTO)
    
    # Títulos y textos
    title1_text = axs['title'].text(0.5, 0.65, jugador_nombre, fontsize=32, 
                                     ha='center', va='center', color=color_jug, weight='bold')
    
    titulo_competicion = f"{jugador['equipo']} | {competicion}"
    title2_text = axs['title'].text(0.5, 0.25, titulo_competicion, 
                                     fontsize=22, ha='center', va='center', color=COLOR_TEXTO)
    
    # Endnote según modo
    if modo == 'percentiles':
        endnote_txt = f'Percentiles vs {posicion} en {competicion} | Min. 300 min | @tu_usuario'
    else:
        endnote_txt = f'Valores reales vs Promedio {posicion} en {competicion} | Min. 300 min | @tu_usuario'
    
    endnote_text = axs['endnote'].text(0.99, 0.5, endnote_txt, 
                                        fontsize=15, ha='right', va='center', color=COLOR_ENDNOTE)
    
    plt.tight_layout()
    
    if guardar:
        plt.savefig(filename, dpi=300, facecolor=COLOR_FONDO, bbox_inches='tight')
        print(f"✅ Gráfico guardado como '{filename}'")
    
    plt.show()
    
    # Mostrar valores
    if modo == 'percentiles':
        print(f"\n📊 Percentiles de {jugador_nombre} ({posicion}):")
        for label, percentil, valor in zip(labels, percentiles, valores_reales):
            print(f"  {label}: {percentil:.1f}º percentil (valor: {valor:.2f})")
    else:
        print(f"\n📊 Valores reales de {jugador_nombre} ({posicion}):")
        for label, valor_jug, valor_prom in zip(labels, valores_jugador, valores_promedio):
            diff = ((valor_jug - valor_prom) / valor_prom * 100) if valor_prom > 0 else 0
            print(f"  {label}: {valor_jug:.2f} vs {valor_prom:.2f} ({diff:+.1f}%)")


# ============================================
# 5. FUNCIÓN RADAR CHART - COMPARACIÓN
# ============================================

def crear_radar_comparacion(jugador1_nombre, jugador2_nombre, competicion='La Liga', 
                            modo='percentiles', guardar=False, filename='radar_comparacion.png',
                            color_jugador1=None, color_jugador2=None):
    """
    Crea un radar chart comparando dos jugadores
    
    Parámetros:
    - modo: 'percentiles' o 'valores_reales'
    - competicion: Incluye opción 'Top 5 Ligas'
    - color_jugador1, color_jugador2: Colores personalizados (opcional)
    """
    # Usar colores personalizados o los por defecto
    color_j1 = color_jugador1 if color_jugador1 else COLOR_JUGADOR
    color_j2 = color_jugador2 if color_jugador2 else COLOR_COMPARACION
    
    # Filtrar por competición
    if competicion == 'Top 5 Ligas':
        df_liga = df.copy()
    else:
        df_liga = df[df['competicion'] == competicion].copy()
    
    # Verificar que ambos jugadores existen
    if jugador1_nombre not in df_liga['jugador'].values:
        print(f"❌ Jugador '{jugador1_nombre}' no encontrado en {competicion}")
        return
    
    if jugador2_nombre not in df_liga['jugador'].values:
        print(f"❌ Jugador '{jugador2_nombre}' no encontrado en {competicion}")
        return
    
    # Obtener posición (asumimos misma posición)
    posicion = df_liga[df_liga['jugador'] == jugador1_nombre]['posicion'].values[0]
    
    # Obtener valores
    valores1, valores2, labels, jugador1, jugador2 = obtener_valores_comparacion(
        df_liga, jugador1_nombre, jugador2_nombre, posicion, modo
    )
    
    # Configurar radar según el modo
    if modo == 'percentiles':
        low = [0] * len(labels)
        high = [100] * len(labels)
    else:  # valores_reales
        # CORRECCIÓN: Limitar valores de porcentajes a 100 y calcular rangos dinámicos
        low = [0] * len(labels)
        high = []
        
        # Clipear valores de porcentajes a 100
        for i, label in enumerate(labels):
            if '%' in label:
                valores1[i] = min(valores1[i], 100)
                valores2[i] = min(valores2[i], 100)
        
        # Calcular rangos
        for i, (label, val1, val2) in enumerate(zip(labels, valores1, valores2)):
            max_val = max(val1, val2)
            if '%' in label:
                high.append(100)
            else:
                high.append(max_val * 1.2)
    
    radar = Radar(labels, low, high,
                  num_rings=4,
                  ring_width=1,
                  center_circle_radius=1)
    
    # Crear figura
    fig, axs = grid(figheight=14, grid_height=0.915, title_height=0.06, endnote_height=0.025,
                    title_space=0, endnote_space=0, grid_key='radar', axis=False)
    
    # Configurar fondo COMPLETO
    fig.patch.set_facecolor(COLOR_FONDO)
    for ax_name in axs:
        axs[ax_name].patch.set_facecolor(COLOR_FONDO)
    
    # Plot radar
    radar.setup_axis(ax=axs['radar'], facecolor=COLOR_FONDO)
    axs['radar'].set_facecolor(COLOR_FONDO)
    
    rings_inner = radar.draw_circles(ax=axs['radar'], facecolor=COLOR_ANILLOS, 
                                     edgecolor=COLOR_BORDE_ANILLOS, linewidth=1.5)
    
    radar_output = radar.draw_radar_compare(valores1, valores2, ax=axs['radar'],
                                            kwargs_radar={'facecolor': color_j1, 'alpha': 0.6, 
                                                        'edgecolor': color_j1, 'linewidth': 2},
                                            kwargs_compare={'facecolor': color_j2, 'alpha': 0.6, 
                                                          'edgecolor': color_j2, 'linewidth': 2})
    
    radar_poly, radar_poly2, vertices1_pos, vertices2_pos = radar_output
    
    # Range labels
    range_labels = radar.draw_range_labels(ax=axs['radar'], fontsize=20, color=COLOR_TEXTO)
    
    # Param labels
    param_labels = radar.draw_param_labels(ax=axs['radar'], fontsize=22, color=COLOR_TEXTO)
    
    # Scatter en vértices
    axs['radar'].scatter(vertices1_pos[:, 0], vertices1_pos[:, 1],
                         c=color_j1, edgecolors=COLOR_TEXTO, marker='o', s=150, zorder=2)
    axs['radar'].scatter(vertices2_pos[:, 0], vertices2_pos[:, 1],
                         c=color_j2, edgecolors=COLOR_TEXTO, marker='o', s=150, zorder=2)
    
    # Títulos
    title1_text = axs['title'].text(0.01, 0.65, jugador1_nombre, fontsize=28,
                                     ha='left', va='center', color=color_j1, weight='bold')
    title2_text = axs['title'].text(0.01, 0.25, jugador1['equipo'], fontsize=20,
                                     ha='left', va='center', color=color_j1)
    
    title3_text = axs['title'].text(0.99, 0.65, jugador2_nombre, fontsize=28,
                                     ha='right', va='center', color=color_j2, weight='bold')
    title4_text = axs['title'].text(0.99, 0.25, jugador2['equipo'], fontsize=20,
                                     ha='right', va='center', color=color_j2)
    
    # Endnote según modo
    if modo == 'percentiles':
        endnote_txt = f'Percentiles en {competicion} | Min. 300 min | @tu_usuario'
    else:
        endnote_txt = f'Valores reales en {competicion} | Min. 300 min | @tu_usuario'
    
    endnote_text = axs['endnote'].text(0.99, 0.5, endnote_txt,
                                        fontsize=15, ha='right', va='center', color=COLOR_ENDNOTE)
    
    plt.tight_layout()
    
    if guardar:
        plt.savefig(filename, dpi=300, facecolor=COLOR_FONDO, bbox_inches='tight')
        print(f"✅ Gráfico guardado como '{filename}'")
    
    plt.show()


# ============================================
# 6. EJEMPLOS DE USO
# ============================================

print("\n" + "="*70)
print("FUNCIONES DISPONIBLES:")
print("="*70)

print("\n1️⃣ RADAR INDIVIDUAL:")
print("   • Percentiles LaLiga:")
print("     crear_radar_individual('Pedri', modo='percentiles')")
print("\n   • Valores reales LaLiga:")
print("     crear_radar_individual('Pedri', modo='valores_reales')")
print("\n   • Percentiles Top 5 Ligas:")
print("     crear_radar_individual('Pedri', competicion='Top 5 Ligas', modo='percentiles')")
print("\n   • Con colores personalizados:")
print("     crear_radar_individual('Pedri', color_jugador='#A50044', color_promedio='#004D98')")

print("\n2️⃣ RADAR COMPARACIÓN:")
print("   • Percentiles:")
print("     crear_radar_comparacion('Pedri', 'Gavi', modo='percentiles')")
print("\n   • Con colores personalizados (ej. colores de equipos):")
print("     crear_radar_comparacion('Pedri', 'Bellingham', color_jugador1='#A50044', color_jugador2='#FFFFFF')")

print("\n" + "="*70)
print("⚙️  PERSONALIZACIÓN DE COLORES:")
print("="*70)
print("Edita las variables al inicio del código:")
print("  COLOR_JUGADOR = '#00f2c1'      # Color jugador principal")
print("  COLOR_COMPARACION = '#ff6b35'  # Color segundo jugador/promedio")

In [ ]:
crear_radar_individual('David Affengruber', competicion='Top 5 Ligas', modo='valores_reales', guardar=False)


In [ ]:
crear_radar_comparacion('David Affengruber', 'Dean Huijsen', modo='percentiles', guardar=False)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import font_manager
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURACIÓN DE COLORES PERSONALIZABLE
# ============================================
COLOR_JUGADOR = '#00f2c1'        # Color barra del jugador - PERSONALIZABLE
COLOR_PROMEDIO = '#ff6b35'       # Color línea de promedio - PERSONALIZABLE
COLOR_FONDO = '#24282a'          # Fondo oscuro
COLOR_TEXTO = '#ffffff'          # Texto blanco
COLOR_TEXTO_SECUNDARIO = '#808080'  # Texto secundario

# ============================================
# 1. CARGAR Y PREPARAR DATOS
# ============================================

# Cargar datos (usar la misma ruta del radar)
df = pd.read_csv(r'C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\fbref\jugadores_campo_2025_2026.csv')
df = df[df['minutos'] >= 300].copy()

# ============================================
# 2. MÉTRICAS REDUCIDAS POR POSICIÓN
# ============================================

METRICAS_REDUCIDAS = {
    'FW': {
        'metricas': ['goles_por90', 'npxg_por90', 'xg_asistencias_por90', 
                     'tiros_a_puerta_por90', 'regates_exitosos', 'toques_area_rival'],
        'labels': ['Goles p90', 'npxG p90', 'xA p90', 
                   'Tiros a puerta p90', 'Regates exitosos', 'Toques área rival']
    },
    'MF': {
        'metricas': ['goles_asistencias_por90', 'xg_asistencias_por90', 'pases_progresivos',
                     'conducciones_progresivas', 'entradas_ganadas', 'recuperaciones'],
        'labels': ['G+A p90', 'xA p90', 'Pases progresivos',
                   'Conducciones progresivas', 'Entradas ganadas', 'Recuperaciones']
    },
    'DF': {
        'metricas': ['entradas_ganadas', 'intercepciones', 'duelos_aereos_ganados',
                     'despejes', 'porc_precision_pase', 'pases_progresivos'],
        'labels': ['Entradas ganadas', 'Intercepciones', 'Duelos aéreos ganados',
                   'Despejes', '% Precisión pase', 'Pases progresivos']
    }
}

# ============================================
# 3. FUNCIÓN PRINCIPAL - BARRAS HORIZONTALES
# ============================================

def crear_barras_horizontales(jugador_nombre, competicion='La Liga', orientacion='cuadrado',
                              guardar=False, filename='barras_metricas.png',
                              color_jugador=None, color_promedio=None):
    """
    Crea un gráfico de barras horizontales con métricas clave del jugador
    
    Parámetros:
    - jugador_nombre: Nombre del jugador
    - competicion: 'La Liga', 'Premier League', etc., o 'Top 5 Ligas'
    - orientacion: 'cuadrado' (1080x1080) o 'vertical' (1080x1920)
    - color_jugador: Color personalizado para barras
    - color_promedio: Color personalizado para línea de promedio
    """
    # Usar colores personalizados o los por defecto
    color_jug = color_jugador if color_jugador else COLOR_JUGADOR
    color_prom = color_promedio if color_promedio else COLOR_PROMEDIO
    
    # Filtrar por competición
    if competicion == 'Top 5 Ligas':
        df_liga = df.copy()
    else:
        df_liga = df[df['competicion'] == competicion].copy()
    
    # Verificar que el jugador existe
    if jugador_nombre not in df_liga['jugador'].values:
        print(f"❌ Jugador '{jugador_nombre}' no encontrado en {competicion}")
        return
    
    # Obtener posición del jugador
    posicion = df_liga[df_liga['jugador'] == jugador_nombre]['posicion'].values[0]
    jugador = df_liga[df_liga['jugador'] == jugador_nombre].iloc[0]
    
    # Filtrar por posición para comparación
    df_posicion = df_liga[df_liga['posicion'] == posicion].copy()
    
    # Obtener métricas según posición
    metricas = METRICAS_REDUCIDAS[posicion]['metricas']
    labels = METRICAS_REDUCIDAS[posicion]['labels']
    
    # Calcular datos
    valores_jugador = []
    valores_promedio = []
    percentiles = []
    rankings = []
    
    for metrica in metricas:
        valor_jug = jugador[metrica]
        promedio = df_posicion[metrica].mean()
        
        # Limitar porcentajes a 100
        if 'porc_' in metrica or '%' in metrica:
            valor_jug = min(valor_jug, 100)
            promedio = min(promedio, 100)
        
        valores_jugador.append(valor_jug)
        valores_promedio.append(promedio)
        
        # Calcular percentil
        percentil = (df_posicion[metrica] <= valor_jug).sum() / len(df_posicion) * 100
        percentiles.append(percentil)
        
        # Calcular ranking
        ranking = (df_posicion[metrica] > valor_jug).sum() + 1
        total_jugadores = len(df_posicion)
        rankings.append(f"{ranking}º/{total_jugadores}")
    
    # Configurar tamaño de figura según orientación
    if orientacion == 'vertical':
        figsize = (10.8, 19.2)  # 1080x1920 proporción
        title_y = 0.98
        bar_section_height = 0.75
    else:  # cuadrado
        figsize = (12, 12)  # 1080x1080 proporción
        title_y = 0.96
        bar_section_height = 0.70
    
    # Crear figura
    fig, ax = plt.subplots(figsize=figsize, facecolor=COLOR_FONDO)
    ax.set_facecolor(COLOR_FONDO)
    
    # Número de métricas
    n_metricas = len(labels)
    y_positions = np.arange(n_metricas)
    
    # Calcular valores máximos para escala
    max_valores = []
    for i, (val_jug, val_prom, label) in enumerate(zip(valores_jugador, valores_promedio, labels)):
        if '%' in label:
            max_valores.append(100)
        else:
            max_valores.append(max(val_jug, val_prom) * 1.15)
    
    # Dibujar barras del jugador
    bars = ax.barh(y_positions, valores_jugador, height=0.6, 
                   color=color_jug, alpha=0.8, zorder=2)
    
    # Dibujar líneas verticales de promedio
    for i, (y_pos, val_prom, max_val) in enumerate(zip(y_positions, valores_promedio, max_valores)):
        # Línea de promedio
        ax.plot([val_prom, val_prom], [y_pos - 0.4, y_pos + 0.4], 
                color=color_prom, linewidth=3, zorder=3)
        
        # Texto del ranking a la derecha
        ranking_text = rankings[i]
        percentil_text = f"{percentiles[i]:.0f}º"
        
        # Posición del texto (a la derecha de la barra o del eje)
        text_x = max(valores_jugador[i], max_val * 0.02)
        ax.text(text_x + max_val * 0.03, y_pos, 
                f"{valores_jugador[i]:.2f}  |  {ranking_text}  ({percentil_text})",
                va='center', ha='left', fontsize=11, color=COLOR_TEXTO, weight='bold')
    
    # Configurar ejes
    ax.set_yticks(y_positions)
    ax.set_yticklabels(labels, fontsize=13, color=COLOR_TEXTO, weight='bold')
    ax.set_xlim(0, None)
    
    # Ajustar límites del eje X por métrica
    for i, max_val in enumerate(max_valores):
        if i == 0:  # Solo necesitamos hacerlo una vez
            pass
    
    # Remover spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_visible(False)
    ax.spines['left'].set_visible(False)
    
    # Configurar grid y ticks
    ax.tick_params(axis='x', colors=COLOR_TEXTO_SECUNDARIO, labelsize=10)
    ax.tick_params(axis='y', length=0)
    ax.grid(axis='x', alpha=0.2, color=COLOR_TEXTO_SECUNDARIO, linestyle='--')
    ax.set_axisbelow(True)
    
    # Título principal
    fig.text(0.5, title_y, jugador_nombre, 
             ha='center', va='top', fontsize=32, color=color_jug, weight='bold')
    
    # Subtítulo
    fig.text(0.5, title_y - 0.04, f"{jugador['equipo']} | {competicion} | {posicion}", 
             ha='center', va='top', fontsize=18, color=COLOR_TEXTO)
    
    # Leyenda
    legend_y = title_y - 0.09
    fig.text(0.15, legend_y, '━', fontsize=20, color=color_jug, weight='bold')
    fig.text(0.18, legend_y, 'Valor Jugador', fontsize=11, color=COLOR_TEXTO, va='center')
    
    fig.text(0.42, legend_y, '━', fontsize=20, color=color_prom, weight='bold')
    fig.text(0.45, legend_y, 'Promedio Liga', fontsize=11, color=COLOR_TEXTO, va='center')
    
    fig.text(0.68, legend_y, 'Formato: Valor | Ranking (Percentil)', 
             fontsize=10, color=COLOR_TEXTO_SECUNDARIO, va='center')
    
    # Nota al pie
    fig.text(0.98, 0.02, f'Min. 300 min jugados | @tu_usuario', 
             ha='right', va='bottom', fontsize=10, color=COLOR_TEXTO_SECUNDARIO)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.90])
    
    if guardar:
        plt.savefig(filename, dpi=300, facecolor=COLOR_FONDO, bbox_inches='tight')
        print(f"✅ Gráfico guardado como '{filename}'")
    
    plt.show()
    
    # Mostrar resumen en consola
    print(f"\n📊 Resumen de {jugador_nombre} ({posicion}) en {competicion}:")
    print("="*60)
    for label, valor, ranking, percentil in zip(labels, valores_jugador, rankings, percentiles):
        print(f"{label:30s}: {valor:6.2f} | {ranking:8s} ({percentil:5.1f}º percentil)")


# ============================================
# 4. EJEMPLOS DE USO
# ============================================

print("\n" + "="*70)
print("FUNCIÓN DE BARRAS HORIZONTALES:")
print("="*70)

print("\n📊 EJEMPLOS DE USO:")
print("\n1. Gráfico cuadrado (post Instagram):")
print("   crear_barras_horizontales('Pedri', orientacion='cuadrado')")

print("\n2. Gráfico vertical (stories Instagram):")
print("   crear_barras_horizontales('Pedri', orientacion='vertical')")

print("\n3. Con Top 5 Ligas:")
print("   crear_barras_horizontales('Pedri', competicion='Top 5 Ligas', orientacion='cuadrado')")

print("\n4. Con colores personalizados:")
print("   crear_barras_horizontales('Pedri', color_jugador='#A50044', color_promedio='#004D98')")

print("\n5. Guardar imagen:")
print("   crear_barras_horizontales('Pedri', guardar=True, filename='pedri_stats.png')")

In [ ]:
crear_barras_horizontales('Pedri', orientacion='cuadrado')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
from adjustText import adjust_text
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURACIÓN DE COLORES PERSONALIZABLE
# ============================================
COLOR_JUGADOR_DESTACADO = '#00f2c1'  # Color jugador principal - PERSONALIZABLE
COLOR_PUNTOS_NORMALES = '#808080'    # Puntos de otros jugadores - PERSONALIZABLE
COLOR_CUADRANTES = '#ff6b35'         # Líneas de cuadrantes - PERSONALIZABLE
COLOR_FONDO = '#24282a'              # Fondo oscuro
COLOR_TEXTO = '#ffffff'              # Texto blanco
COLOR_TEXTO_SECUNDARIO = '#808080'   # Texto secundario

# ============================================
# 1. CARGAR Y PREPARAR DATOS
# ============================================

# Cargar datos
df = pd.read_csv(r'C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\fbref\jugadores_campo_2025_2026.csv')
df = df[df['minutos'] >= 450].copy()

# ============================================
# 2. MÉTRICAS DISPONIBLES POR POSICIÓN
# ============================================

METRICAS_SCATTER = {
    'FW': {
        'ofensivas': ['goles_por90', 'npxg_por90', 'xg_asistencias_por90', 'tiros_por90', 
                     'tiros_a_puerta_por90', 'goles_asistencias_por90'],
        'creacion': ['pases_progresivos', 'pases_tercio_final', 'xa_modelado', 
                    'tiros_asistidos', 'pases_area'],
        'individuales': ['regates_exitosos', 'conducciones_progresivas', 
                        'toques_area_rival', 'conducciones_area']
    },
    'MF': {
        'creacion': ['xg_asistencias_por90', 'pases_progresivos', 'pases_tercio_final',
                    'xa_modelado', 'tiros_asistidos', 'conducciones_progresivas'],
        'ofensivas': ['goles_asistencias_por90', 'goles_por90', 'npxg_por90', 'tiros_por90'],
        'defensivas': ['entradas_ganadas', 'intercepciones', 'recuperaciones', 
                      'entradas_mas_intercepciones', 'duelos_aereos_ganados']
    },
    'DF': {
        'defensivas': ['entradas_ganadas', 'intercepciones', 'duelos_aereos_ganados',
                      'despejes', 'bloqueos', 'entradas_mas_intercepciones'],
        'construccion': ['pases_progresivos', 'porc_precision_pase', 'pases_largos_completados',
                        'dist_progresiva_pases', 'pases_tercio_final'],
        'aereas': ['duelos_aereos_ganados', 'porc_duelos_aereos_ganados', 'despejes']
    }
}

# ============================================
# 3. FUNCIÓN PRINCIPAL - SCATTER PLOT
# ============================================

def crear_scatter_plot(metrica_x, metrica_y, posicion, competicion='La Liga',
                      jugadores_destacados=None, mostrar_nombres=True,
                      cuadrantes=True, percentil_minimo=75,
                      guardar=False, filename='scatter_plot.png',
                      color_destacado=None, tamano_figura='cuadrado'):
    """
    Crea un scatter plot comparando jugadores en dos métricas
    
    Parámetros:
    - metrica_x: Métrica para eje X (nombre de columna)
    - metrica_y: Métrica para eje Y (nombre de columna)
    - posicion: 'FW', 'MF', o 'DF'
    - competicion: Liga o 'Top 5 Ligas'
    - jugadores_destacados: Lista de nombres de jugadores a destacar (None = Top percentil)
    - mostrar_nombres: Si True, muestra nombres de jugadores destacados
    - cuadrantes: Si True, dibuja líneas de promedio para crear cuadrantes
    - percentil_minimo: Percentil mínimo para destacar jugadores automáticamente (si jugadores_destacados=None)
    - color_destacado: Color personalizado para jugadores destacados
    - tamano_figura: 'cuadrado' (1080x1080) o 'vertical' (1080x1920)
    """
    
    # Usar color personalizado o el por defecto
    color_dest = color_destacado if color_destacado else COLOR_JUGADOR_DESTACADO
    
    # Filtrar por competición y posición
    if competicion == 'Top 5 Ligas':
        df_filtrado = df[df['posicion'] == posicion].copy()
    else:
        df_filtrado = df[(df['competicion'] == competicion) & (df['posicion'] == posicion)].copy()
    
    if len(df_filtrado) == 0:
        print(f"❌ No hay jugadores de posición {posicion} en {competicion}")
        return
    
    # Verificar que las métricas existen
    if metrica_x not in df_filtrado.columns:
        print(f"❌ Métrica '{metrica_x}' no encontrada")
        return
    if metrica_y not in df_filtrado.columns:
        print(f"❌ Métrica '{metrica_y}' no encontrada")
        return
    
    # Obtener datos
    x_data = df_filtrado[metrica_x].values
    y_data = df_filtrado[metrica_y].values
    nombres = df_filtrado['jugador'].values
    
    # Calcular promedios para cuadrantes
    x_promedio = np.mean(x_data)
    y_promedio = np.mean(y_data)
    
    # Determinar jugadores a destacar
    if jugadores_destacados is None:
        # Destacar jugadores en percentil alto en ambas métricas
        percentil_x = np.percentile(x_data, percentil_minimo)
        percentil_y = np.percentile(y_data, percentil_minimo)
        mask_destacados = (x_data >= percentil_x) | (y_data >= percentil_y)
        jugadores_destacados = nombres[mask_destacados].tolist()
    else:
        # Usar lista proporcionada
        mask_destacados = np.isin(nombres, jugadores_destacados)
    
    # Configurar tamaño de figura
    if tamano_figura == 'vertical':
        figsize = (10.8, 19.2)
        title_y = 0.98
    else:  # cuadrado
        figsize = (14, 14)
        title_y = 0.96
    
    # Crear figura
    fig, ax = plt.subplots(figsize=figsize, facecolor=COLOR_FONDO)
    ax.set_facecolor(COLOR_FONDO)
    
    # Plotear puntos normales
    ax.scatter(x_data[~mask_destacados], y_data[~mask_destacados], 
              s=100, color=COLOR_PUNTOS_NORMALES, alpha=0.4, zorder=2, edgecolors='none')
    
    # Plotear puntos destacados
    ax.scatter(x_data[mask_destacados], y_data[mask_destacados], 
              s=200, color=color_dest, alpha=0.8, zorder=3, edgecolors=COLOR_TEXTO, linewidth=1.5)
    
    # Dibujar cuadrantes si está activado
    if cuadrantes:
        # Líneas de cuadrantes con labels de valor
        ax.axvline(x_promedio, color=COLOR_CUADRANTES, linestyle='--', linewidth=2, alpha=0.5, zorder=1)
        ax.axhline(y_promedio, color=COLOR_CUADRANTES, linestyle='--', linewidth=2, alpha=0.5, zorder=1)
        
        # Añadir valor del promedio en las líneas
        # Valor en eje X (arriba)
        ax.text(x_promedio, ax.get_ylim()[1], f' {x_promedio:.2f}', 
                fontsize=10, color=COLOR_CUADRANTES, ha='center', va='bottom', weight='bold')
        
        # Valor en eje Y (derecha)
        ax.text(ax.get_xlim()[1], y_promedio, f' {y_promedio:.2f}', 
                fontsize=10, color=COLOR_CUADRANTES, ha='left', va='center', weight='bold')
    
    # Añadir nombres de jugadores destacados con posicionamiento inteligente
    if mostrar_nombres:
        # Calcular rangos para offset proporcional
        x_range = ax.get_xlim()[1] - ax.get_xlim()[0]
        y_range = ax.get_ylim()[1] - ax.get_ylim()[0]
        offset_x = x_range * 0.02
        offset_y = y_range * 0.04
        
        texts = []
        for i, (nombre, x, y) in enumerate(zip(nombres[mask_destacados], 
                                               x_data[mask_destacados], 
                                               y_data[mask_destacados])):
            # Determinar si está por encima o debajo del promedio en Y
            if y > y_promedio:
                # Punto arriba del promedio -> texto arriba
                va_align = 'bottom'
                y_offset = offset_y
            else:
                # Punto abajo del promedio -> texto abajo
                va_align = 'top'
                y_offset = -offset_y
            
            # Determinar alineación horizontal
            if x > x_promedio:
                ha_align = 'left'
                x_offset = offset_x
            else:
                ha_align = 'right'
                x_offset = -offset_x
            
            # Añadir texto con offset
            texto = ax.text(x + x_offset, y + y_offset, nombre, 
                          fontsize=10, color=COLOR_TEXTO, weight='bold',
                          ha=ha_align, va=va_align, zorder=4)
            
            # Dibujar línea de conexión desde el punto al texto
            ax.annotate('', xy=(x, y), xytext=(x + x_offset, y + y_offset),
                       arrowprops=dict(arrowstyle='-', color=color_dest, lw=1, alpha=0.5),
                       zorder=3)
            
            texts.append(texto)
        
        # Ajustar textos para evitar solapamiento solo si es necesario
        try:
            adjust_text(texts, ax=ax, 
                       only_move={'points': 'y', 'texts': 'xy'},
                       arrowprops=dict(arrowstyle='-', color=color_dest, lw=1, alpha=0.5),
                       expand_points=(1.2, 1.2), expand_text=(1.1, 1.1),
                       force_text=(0.5, 0.5), force_points=(0.2, 0.2))
        except:
            pass  # Si adjustText no está disponible o falla, los textos quedan posicionados
    
    # Configurar ejes
    ax.set_xlabel(metrica_x.replace('_', ' ').title(), fontsize=16, color=COLOR_TEXTO, weight='bold')
    ax.set_ylabel(metrica_y.replace('_', ' ').title(), fontsize=16, color=COLOR_TEXTO, weight='bold')
    
    ax.tick_params(axis='both', colors=COLOR_TEXTO, labelsize=12)
    ax.grid(True, alpha=0.2, color=COLOR_TEXTO_SECUNDARIO, linestyle='-', linewidth=0.5)
    
    # Spines
    for spine in ax.spines.values():
        spine.set_edgecolor(COLOR_TEXTO_SECUNDARIO)
        spine.set_linewidth(1.5)
    
    # Título
    fig.text(0.5, title_y, f'{posicion} - {competicion}', 
             ha='center', va='top', fontsize=28, color=COLOR_TEXTO, weight='bold')
    
    # Subtítulo con métricas
    fig.text(0.5, title_y - 0.04, 
             f'{metrica_x.replace("_", " ").title()} vs {metrica_y.replace("_", " ").title()}', 
             ha='center', va='top', fontsize=18, color=COLOR_TEXTO_SECUNDARIO)
    
    # Información adicional - MOVIDA ABAJO
    n_jugadores = len(df_filtrado)
    n_destacados = mask_destacados.sum()
    fig.text(0.5, 0.00, 
             f'{n_jugadores} jugadores analizados | {n_destacados} destacados', 
             ha='center', va='bottom', fontsize=12, color=COLOR_TEXTO_SECUNDARIO)
    
    # Nota al pie
    fig.text(0.98, 0.02, f'Min. 450 min jugados | @manuelperezruda', 
             ha='right', va='bottom', fontsize=10, color=COLOR_TEXTO_SECUNDARIO)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.90])
    
    if guardar:
        plt.savefig(filename, dpi=300, facecolor=COLOR_FONDO, bbox_inches='tight')
        print(f"✅ Gráfico guardado como '{filename}'")
    
    plt.show()
    
    # Mostrar top 5 en ambas métricas
    df_filtrado['score_combinado'] = (
        (df_filtrado[metrica_x] - df_filtrado[metrica_x].mean()) / df_filtrado[metrica_x].std() +
        (df_filtrado[metrica_y] - df_filtrado[metrica_y].mean()) / df_filtrado[metrica_y].std()
    )
    top5 = df_filtrado.nlargest(5, 'score_combinado')[['jugador', 'equipo', metrica_x, metrica_y]]
    
    print(f"\n📊 Top 5 jugadores (score combinado):")
    print("="*70)
    for idx, row in top5.iterrows():
        print(f"{row['jugador']:25s} ({row['equipo']:15s}): {metrica_x}={row[metrica_x]:.2f}, {metrica_y}={row[metrica_y]:.2f}")


# ============================================
# 4. FUNCIÓN AUXILIAR - VER MÉTRICAS DISPONIBLES
# ============================================

def ver_metricas_disponibles(posicion):
    """
    Muestra las métricas disponibles para scatter plot según posición
    """
    if posicion not in METRICAS_SCATTER:
        print(f"❌ Posición '{posicion}' no válida. Usa: 'FW', 'MF', o 'DF'")
        return
    
    print(f"\n📊 MÉTRICAS DISPONIBLES PARA {posicion}:")
    print("="*70)
    
    for categoria, metricas in METRICAS_SCATTER[posicion].items():
        print(f"\n{categoria.upper()}:")
        for metrica in metricas:
            print(f"  • {metrica}")
    
    print("\n💡 También puedes usar cualquier columna del dataset como métrica.")


# ============================================
# 5. EJEMPLOS DE USO
# ============================================

print("\n" + "="*70)
print("SCATTER PLOT - COMPARACIÓN MULTI-JUGADOR")
print("="*70)

print("\n📊 EJEMPLOS DE USO:")

print("\n1️⃣ Scatter básico (destacar top percentil 75):")
print("   crear_scatter_plot('goles_por90', 'npxg_por90', 'FW', competicion='La Liga')")

print("\n2️⃣ Destacar jugadores específicos:")
print("   crear_scatter_plot('pases_progresivos', 'xg_asistencias_por90', 'MF',")
print("                      jugadores_destacados=['Pedri', 'Gavi', 'Bellingham'])")

print("\n3️⃣ Sin cuadrantes y solo nombres destacados:")
print("   crear_scatter_plot('entradas_ganadas', 'intercepciones', 'DF',")
print("                      cuadrantes=False, percentil_minimo=90)")

print("\n4️⃣ Con Top 5 Ligas:")
print("   crear_scatter_plot('goles_por90', 'tiros_a_puerta_por90', 'FW',")
print("                      competicion='Top 5 Ligas')")

print("\n5️⃣ Color personalizado:")
print("   crear_scatter_plot('regates_exitosos', 'conducciones_progresivas', 'FW',")
print("                      color_destacado='#A50044')")

print("\n" + "="*70)
print("💡 Para ver métricas disponibles por posición:")
print("="*70)
print("   ver_metricas_disponibles('FW')  # o 'MF' o 'DF'")

In [ ]:
crear_scatter_plot('goles_por90', 'xg_por90', 'FW', competicion='Top 5 Ligas', percentil_minimo=90)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURACIÓN DE COLORES PERSONALIZABLE
# ============================================
COLOR_PRINCIPAL = '#00f2c1'         # Color de barras y acentos - PERSONALIZABLE
COLOR_FONDO = '#24282a'             # Fondo oscuro
COLOR_FILA_ALT = '#2d3235'          # Color filas alternas
COLOR_TEXTO = '#ffffff'             # Texto blanco
COLOR_TEXTO_SECUNDARIO = '#808080'  # Texto secundario
COLOR_BORDE = '#3a3f42'             # Bordes de tabla

# ============================================
# 1. CARGAR Y PREPARAR DATOS
# ============================================

# Cargar datos
df = pd.read_csv(r'C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\fbref\jugadores_campo_2025_2026.csv')
df = df[df['minutos'] >= 400].copy()

# ============================================
# 2. FUNCIÓN PRINCIPAL - TABLA RANKING
# ============================================

def crear_tabla_ranking(metrica, posicion=None, competicion='La Liga', top_n=10,
                       titulo_personalizado=None, orden_descendente=True,
                       guardar=False, filename='tabla_ranking.png',
                       color_principal=None, orientacion='cuadrado'):
    """
    Crea una tabla de ranking de jugadores según una métrica
    
    Parámetros:
    - metrica: Métrica a rankear (nombre de columna)
    - posicion: 'FW', 'MF', 'DF', o None para todas las posiciones
    - competicion: Liga o 'Top 5 Ligas'
    - top_n: Número de jugadores a mostrar (top 5, 10, 15, 20...)
    - titulo_personalizado: Título custom (None = auto)
    - orden_descendente: True para mayor a menor, False para menor a mayor
    - color_principal: Color personalizado para barras
    - orientacion: 'cuadrado' (1080x1080) o 'vertical' (1080x1920)
    """
    
    # Usar color personalizado o el por defecto
    color = color_principal if color_principal else COLOR_PRINCIPAL
    
    # Filtrar por competición
    if competicion == 'Top 5 Ligas':
        df_filtrado = df.copy()
    else:
        df_filtrado = df[df['competicion'] == competicion].copy()
    
    # Filtrar por posición si se especifica
    if posicion:
        df_filtrado = df_filtrado[df_filtrado['posicion'] == posicion].copy()
        texto_posicion = f" ({posicion})"
    else:
        texto_posicion = ""
    
    if len(df_filtrado) == 0:
        print(f"No hay datos para la combinación especificada")
        return
    
    # Verificar que la métrica existe
    if metrica not in df_filtrado.columns:
        print(f"Métrica '{metrica}' no encontrada")
        return
    
    # Ordenar y obtener top N
    df_top = df_filtrado.nlargest(top_n, metrica) if orden_descendente else df_filtrado.nsmallest(top_n, metrica)
    
    # Preparar datos
    rankings = list(range(1, len(df_top) + 1))
    jugadores = df_top['jugador'].values
    equipos = df_top['equipo'].values
    valores = df_top[metrica].values
    
    # Limitar porcentajes a 100
    if 'porc_' in metrica or '%' in metrica:
        valores = np.minimum(valores, 100)
    
    # Configurar tamaño de figura
    if orientacion == 'vertical':
        figsize = (10.8, 19.2)
        row_height = 0.8
    else:  # cuadrado
        figsize = (14, 14)
        row_height = 0.9
    
    # Crear figura
    fig, ax = plt.subplots(figsize=figsize, facecolor=COLOR_FONDO)
    ax.set_facecolor(COLOR_FONDO)
    
    # Ocultar ejes
    ax.axis('off')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, top_n + 3)
    
    # Título - PERSONALIZABLE: Color del título (color)
    if titulo_personalizado:
        titulo = titulo_personalizado
    else:
        titulo = f"Top {top_n} - {metrica.replace('_', ' ').title()}{texto_posicion}"
    
    ax.text(5, top_n + 2.3, titulo, ha='center', va='top', 
            fontsize=28, color=color, weight='bold')
    
    # Subtítulo - PERSONALIZABLE: Color del subtítulo (COLOR_TEXTO_SECUNDARIO)
    ax.text(5, top_n + 1.8, competicion, ha='center', va='top',
            fontsize=18, color=COLOR_TEXTO_SECUNDARIO)
    
    # Headers de columnas
    header_y = top_n + 1.2
    # PERSONALIZABLE: Color de headers (COLOR_TEXTO_SECUNDARIO)
    ax.text(0.5, header_y, '#', ha='center', va='center', 
            fontsize=13, color=COLOR_TEXTO_SECUNDARIO, weight='bold')
    ax.text(2.0, header_y, 'JUGADOR', ha='left', va='center',
            fontsize=13, color=COLOR_TEXTO_SECUNDARIO, weight='bold')
    ax.text(5.0, header_y, 'EQUIPO', ha='left', va='center',
            fontsize=13, color=COLOR_TEXTO_SECUNDARIO, weight='bold')
    ax.text(9.5, header_y, metrica.replace('_', ' ').upper(), ha='right', va='center',
            fontsize=13, color=COLOR_TEXTO_SECUNDARIO, weight='bold')
    
    # Línea debajo del header - PERSONALIZABLE: Color del borde (COLOR_BORDE)
    ax.plot([0.2, 9.8], [header_y - 0.25, header_y - 0.25], 
            color=COLOR_BORDE, linewidth=2)
    
    # Calcular valor máximo para escala de barras
    max_valor = valores.max()
    
    # Dibujar filas
    for i, (rank, jugador, equipo, valor) in enumerate(zip(rankings, jugadores, equipos, valores)):
        y_pos = top_n - i
        
        # Fondo alternado para filas
        if i % 2 == 0:
            # PERSONALIZABLE: Color de filas alternas (COLOR_FILA_ALT)
            rect = plt.Rectangle((0.2, y_pos - row_height/2), 9.6, row_height,
                                facecolor=COLOR_FILA_ALT, edgecolor='none', zorder=1)
            ax.add_patch(rect)
        
        # Ranking - PERSONALIZABLE: color del ranking top 3 (color) vs resto (COLOR_TEXTO)
        rank_color = color if rank <= 3 else COLOR_TEXTO
        rank_weight = 'bold' if rank <= 3 else 'normal'
        ax.text(0.5, y_pos, str(rank), ha='center', va='center',
                fontsize=14, color=rank_color, weight=rank_weight, zorder=3)
        
        # Jugador - PERSONALIZABLE: Color del nombre del jugador (COLOR_TEXTO)
        ax.text(2.0, y_pos, jugador, ha='left', va='center',
                fontsize=12, color=COLOR_TEXTO, weight='bold', zorder=3)
        
        # Equipo - PERSONALIZABLE: Color del equipo (COLOR_TEXTO_SECUNDARIO)
        ax.text(5.0, y_pos, equipo, ha='left', va='center',
                fontsize=11, color=COLOR_TEXTO_SECUNDARIO, zorder=3)
        
        # Barra de valor (proporcional) - Más ancha para mayor presencia
        # PERSONALIZABLE: Color de barras (color) y transparencia (alpha)
        bar_width = (valor / max_valor) * 2.5  # Aumentado de 1.5 a 2.5
        bar = plt.Rectangle((7.0, y_pos - 0.25), bar_width, 0.5,
                           facecolor=color, alpha=0.6, edgecolor='none', zorder=2)
        ax.add_patch(bar)
        
        # Valor numérico - SIN DECIMALES
        # PERSONALIZABLE: Color de los valores numéricos (COLOR_TEXTO)
        ax.text(9.5, y_pos, f'{int(valor)}', ha='right', va='center',
                fontsize=12, color=COLOR_TEXTO, weight='bold', zorder=3)
    
    # Nota al pie
    ax.text(5, 0.3, f'Min. 400 min jugados | {len(df_filtrado)} jugadores analizados | Datos: FBref (Opta) | @manuelperezruda',
            ha='center', va='center', fontsize=10, color=COLOR_TEXTO_SECUNDARIO)
    
    plt.tight_layout()
    
    if guardar:
        plt.savefig(filename, dpi=300, facecolor=COLOR_FONDO, bbox_inches='tight')
        print(f"Gráfico guardado como '{filename}'")
    
    plt.show()
    
    # Mostrar resumen en consola
    print(f"\nTop {top_n} - {metrica} en {competicion}{texto_posicion}:")
    print("="*70)
    for rank, jugador, equipo, valor in zip(rankings, jugadores, equipos, valores):
        print(f"{rank:2d}. {jugador:25s} ({equipo:15s}): {valor:.2f}")


# ============================================
# 3. EJEMPLOS DE USO
# ============================================

print("\n" + "="*70)
print("TABLA DE RANKING - TOP JUGADORES")
print("="*70)

print("\nEJEMPLOS DE USO:")

print("\n1️⃣ FILTRADO POR POSICIÓN:")
print("   crear_tabla_ranking('goles_por90', posicion='FW', top_n=10)")
print("   crear_tabla_ranking('pases_progresivos', posicion='MF', top_n=15)")

print("\n2️⃣ SIN FILTRO DE POSICIÓN (todas las posiciones):")
print("   crear_tabla_ranking('recuperaciones', posicion=None, top_n=20)")
print("   crear_tabla_ranking('conducciones_progresivas', posicion=None, top_n=15)")
print("   crear_tabla_ranking('duelos_aereos_ganados', posicion=None, top_n=10)")

print("\n3️⃣ Top 5 Ligas:")
print("   crear_tabla_ranking('goles_por90', posicion='FW', competicion='Top 5 Ligas', top_n=20)")

print("\n4️⃣ Con título personalizado:")
print("   crear_tabla_ranking('goles_por90', posicion='FW', titulo_personalizado='Los Goleadores de LaLiga')")

print("\n5️⃣ Orden ascendente (para métricas donde menos es mejor):")
print("   crear_tabla_ranking('errores', posicion='DF', orden_descendente=False)")

print("\n6️⃣ Color personalizado:")
print("   crear_tabla_ranking('goles_por90', posicion='FW', color_principal='#A50044')")

print("\n7️⃣ Formato vertical (stories):")
print("   crear_tabla_ranking('goles_por90', posicion='FW', orientacion='vertical', top_n=15)")

In [ ]:
crear_tabla_ranking('recuperaciones', posicion=None, top_n=10)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from matplotlib import font_manager
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURACIÓN DE COLORES PERSONALIZABLE
# ============================================
COLOR_JUGADOR = '#00f2c1'           # Color línea del jugador - PERSONALIZABLE
COLOR_DENSIDAD = '#ff6b35'          # Color curva de densidad - PERSONALIZABLE
COLOR_AREA = '#ff6b35'              # Color área bajo la curva - PERSONALIZABLE
COLOR_PROMEDIO = '#ffffff'          # Color línea de promedio - PERSONALIZABLE
COLOR_FONDO = '#24282a'             # Fondo oscuro
COLOR_TEXTO = '#ffffff'             # Texto blanco
COLOR_TEXTO_SECUNDARIO = '#808080'  # Texto secundario

# ============================================
# 1. CARGAR Y PREPARAR DATOS
# ============================================

# Cargar datos
df = pd.read_csv(r'C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\fbref\jugadores_campo_2025_2026.csv')
df = df[df['minutos'] >= 400].copy()

# ============================================
# 2. FUNCIÓN PRINCIPAL - GRÁFICO DE DENSIDAD
# ============================================

def crear_grafico_densidad(metrica, jugador_nombre, posicion=None, competicion='La Liga',
                          titulo_personalizado=None, guardar=False, filename='densidad.png',
                          color_jugador=None, color_densidad=None, orientacion='cuadrado'):
    """
    Crea un gráfico de densidad mostrando la distribución de una métrica
    y destacando la posición de un jugador específico
    
    Parámetros:
    - metrica: Métrica a analizar (nombre de columna)
    - jugador_nombre: Nombre del jugador a destacar
    - posicion: 'FW', 'MF', 'DF', o None para todas las posiciones
    - competicion: Liga o 'Top 5 Ligas'
    - titulo_personalizado: Título custom (None = auto)
    - color_jugador: Color personalizado para línea del jugador
    - color_densidad: Color personalizado para curva de densidad
    - orientacion: 'cuadrado' (1080x1080) o 'vertical' (1080x1920)
    """
    
    # Usar colores personalizados o los por defecto
    color_jug = color_jugador if color_jugador else COLOR_JUGADOR
    color_dens = color_densidad if color_densidad else COLOR_DENSIDAD
    
    # Filtrar por competición
    if competicion == 'Top 5 Ligas':
        df_filtrado = df.copy()
    else:
        df_filtrado = df[df['competicion'] == competicion].copy()
    
    # Filtrar por posición si se especifica
    if posicion:
        df_filtrado = df_filtrado[df_filtrado['posicion'] == posicion].copy()
        texto_posicion = f" ({posicion})"
    else:
        texto_posicion = ""
    
    # Verificar que el jugador existe
    if jugador_nombre not in df_filtrado['jugador'].values:
        print(f"❌ Jugador '{jugador_nombre}' no encontrado")
        return
    
    # Verificar que la métrica existe
    if metrica not in df_filtrado.columns:
        print(f"❌ Métrica '{metrica}' no encontrada")
        return
    
    # Obtener datos
    datos = df_filtrado[metrica].values
    
    # Limitar porcentajes a 100
    if 'porc_' in metrica or '%' in metrica:
        datos = np.minimum(datos, 100)
    
    # Obtener valor del jugador
    valor_jugador = df_filtrado[df_filtrado['jugador'] == jugador_nombre][metrica].values[0]
    if 'porc_' in metrica or '%' in metrica:
        valor_jugador = min(valor_jugador, 100)
    
    equipo_jugador = df_filtrado[df_filtrado['jugador'] == jugador_nombre]['equipo'].values[0]
    
    # Calcular estadísticas
    promedio = np.mean(datos)
    mediana = np.median(datos)
    percentil = (datos <= valor_jugador).sum() / len(datos) * 100
    ranking = (datos > valor_jugador).sum() + 1
    
    # Configurar tamaño de figura
    if orientacion == 'vertical':
        figsize = (10.8, 19.2)
    else:  # cuadrado
        figsize = (14, 14)
    
    # Crear figura con layout más compacto
    fig = plt.figure(figsize=figsize, facecolor=COLOR_FONDO)
    
    # Crear grid de subplots: gráfico arriba (70%), info abajo (30%)
    gs = fig.add_gridspec(2, 1, height_ratios=[2.5, 1], hspace=0.15)
    
    # Subplot superior: gráfico de densidad
    ax = fig.add_subplot(gs[0])
    ax.set_facecolor(COLOR_FONDO)
    
    # Calcular densidad usando KDE
    try:
        kde = stats.gaussian_kde(datos)
        x_range = np.linspace(datos.min(), datos.max(), 1000)
        density = kde(x_range)
        
        # Dibujar curva de densidad - PERSONALIZABLE: color y grosor
        ax.plot(x_range, density, color=color_dens, linewidth=2.5, label='Distribución', zorder=2)
        
        # Rellenar área bajo la curva - PERSONALIZABLE: color y transparencia
        ax.fill_between(x_range, density, alpha=0.3, color=color_dens, zorder=1)
        
        # Sombrear área donde el jugador supera a otros
        mask = x_range <= valor_jugador
        ax.fill_between(x_range[mask], density[mask], alpha=0.5, color=color_jug, zorder=1)
        
    except Exception as e:
        print(f"No se pudo calcular la densidad: {e}")
        return
    
    # Línea vertical del jugador - PERSONALIZABLE: color, grosor y estilo
    ax.axvline(valor_jugador, color=color_jug, linewidth=3.5, linestyle='-', 
              label=jugador_nombre, zorder=3, alpha=0.9)
    
    # Línea vertical del promedio - PERSONALIZABLE: color, grosor y estilo
    ax.axvline(promedio, color=COLOR_PROMEDIO, linewidth=2, linestyle='--', 
              label=f'Promedio', zorder=3, alpha=0.5)
    
    # Configurar ejes - PERSONALIZABLE: colores de texto
    ax.set_xlabel(metrica.replace('_', ' ').title(), fontsize=14, 
                 color=COLOR_TEXTO, weight='bold')
    ax.set_ylabel('Densidad', fontsize=14, color=COLOR_TEXTO, weight='bold')
    
    ax.tick_params(axis='both', colors=COLOR_TEXTO, labelsize=11)
    ax.grid(True, alpha=0.2, color=COLOR_TEXTO_SECUNDARIO, linestyle='-', linewidth=0.5)
    
    # Spines - PERSONALIZABLE: color de bordes
    for spine in ax.spines.values():
        spine.set_edgecolor(COLOR_TEXTO_SECUNDARIO)
        spine.set_linewidth(1.5)
    
    # Subplot inferior: información del jugador
    ax_info = fig.add_subplot(gs[1])
    ax_info.set_facecolor(COLOR_FONDO)
    ax_info.axis('off')
    ax_info.set_xlim(0, 10)
    ax_info.set_ylim(0, 4)
    
    # Nombre del jugador - PERSONALIZABLE: color del nombre
    ax_info.text(5, 3.2, jugador_nombre, ha='center', va='center',
                fontsize=22, color=color_jug, weight='bold')
    
    # Equipo - PERSONALIZABLE: color del equipo
    ax_info.text(5, 2.6, equipo_jugador, ha='center', va='center',
                fontsize=14, color=COLOR_TEXTO)
    
    # Línea separadora
    ax_info.plot([1, 9], [2.1, 2.1], color=COLOR_TEXTO_SECUNDARIO, linewidth=1, alpha=0.3)
    
    # Estadísticas en tres columnas - PERSONALIZABLE: colores de las estadísticas
    # Columna 1: Valor
    ax_info.text(2, 1.4, 'VALOR', ha='center', va='center',
                fontsize=11, color=COLOR_TEXTO_SECUNDARIO, weight='bold')
    ax_info.text(2, 0.7, f'{valor_jugador:.2f}', ha='center', va='center',
                fontsize=18, color=color_jug, weight='bold')
    
    # Columna 2: Ranking
    ax_info.text(5, 1.4, 'RANKING', ha='center', va='center',
                fontsize=11, color=COLOR_TEXTO_SECUNDARIO, weight='bold')
    ax_info.text(5, 0.7, f'{ranking}º / {len(datos)}', ha='center', va='center',
                fontsize=18, color=color_jug, weight='bold')
    
    # Columna 3: Percentil
    ax_info.text(8, 1.4, 'PERCENTIL', ha='center', va='center',
                fontsize=11, color=COLOR_TEXTO_SECUNDARIO, weight='bold')
    ax_info.text(8, 0.7, f'{percentil:.0f}º', ha='center', va='center',
                fontsize=18, color=color_jug, weight='bold')
    
    # Título principal en la parte superior - PERSONALIZABLE: color del título
    if titulo_personalizado:
        titulo = titulo_personalizado
    else:
        titulo = f"{metrica.replace('_', ' ').title()}{texto_posicion}"
    
    fig.text(0.5, 0.96, titulo, ha='center', va='top', 
            fontsize=26, color=color_dens, weight='bold')
    
    # Subtítulo - PERSONALIZABLE: color del subtítulo
    fig.text(0.5, 0.92, competicion, ha='center', va='top',
            fontsize=16, color=COLOR_TEXTO_SECUNDARIO)
    
    # Nota al pie - PERSONALIZABLE: color de la nota
    fig.text(0.5, 0.02, f'Min. 400 min jugados | {len(datos)} jugadores analizados | Datos: FBref (Opta) | @manuelperezruda',
            ha='center', va='center', fontsize=9, color=COLOR_TEXTO_SECUNDARIO)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.90])
    
    if guardar:
        plt.savefig(filename, dpi=300, facecolor=COLOR_FONDO, bbox_inches='tight')
        print(f"✅ Gráfico guardado como '{filename}'")
    
    plt.show()
    
    # Mostrar resumen en consola
    print(f"\n📊 Análisis de densidad - {jugador_nombre}:")
    print("="*70)
    print(f"Métrica: {metrica}")
    print(f"Valor del jugador: {valor_jugador:.2f}")
    print(f"Promedio de la liga: {promedio:.2f}")
    print(f"Ranking: {ranking}º de {len(datos)} jugadores")
    print(f"Percentil: {percentil:.1f}º")
    print(f"Diferencia vs promedio: {((valor_jugador - promedio) / promedio * 100):+.1f}%")


# ============================================
# 3. EJEMPLOS DE USO
# ============================================

print("\n" + "="*70)
print("GRÁFICO DE DENSIDAD - DISTRIBUCIÓN DE MÉTRICA")
print("="*70)

print("\n📊 EJEMPLOS DE USO:")

print("\n1️⃣ Densidad básica (filtrado por posición):")
print("   crear_grafico_densidad('goles_por90', 'Robert Lewandowski', posicion='FW')")

print("\n2️⃣ Sin filtro de posición:")
print("   crear_grafico_densidad('recuperaciones', 'Pedri', posicion=None)")

print("\n3️⃣ Top 5 Ligas:")
print("   crear_grafico_densidad('goles_por90', 'Robert Lewandowski', posicion='FW', competicion='Top 5 Ligas')")

print("\n4️⃣ Con título personalizado:")
print("   crear_grafico_densidad('goles_por90', 'Lewandowski', posicion='FW',")
print("                          titulo_personalizado='Capacidad Goleadora')")

print("\n5️⃣ Colores personalizados:")
print("   crear_grafico_densidad('goles_por90', 'Lewandowski', posicion='FW',")
print("                          color_jugador='#A50044', color_densidad='#FFED02')")

print("\n6️⃣ Formato vertical (stories):")
print("   crear_grafico_densidad('goles_por90', 'Lewandowski', posicion='FW', orientacion='vertical')")

In [ ]:
crear_grafico_densidad('goles_por90', 'Ferrán Torres', posicion='FW',
                          titulo_personalizado='Capacidad Goleadora')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import font_manager
import unicodedata
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURACIÓN DE COLORES PERSONALIZABLE
# ============================================
# PERSONALIZABLE: Colores del gradiente del heatmap (de bajo a alto)
COLOR_BAJO = '#1a1d1f'              # Muy bajo - casi negro
COLOR_MEDIO_BAJO = '#ff6b35'        # Medio-bajo - naranja
COLOR_MEDIO = '#ffb547'             # Medio - amarillo
COLOR_MEDIO_ALTO = '#a0e8af'        # Medio-alto - verde claro
COLOR_ALTO = '#00f2c1'              # Alto - cyan

COLOR_FONDO = '#24282a'             # Fondo oscuro
COLOR_TEXTO = '#ffffff'             # Texto blanco
COLOR_TEXTO_SECUNDARIO = '#808080'  # Texto secundario
COLOR_GRID = '#3a3f42'              # Líneas de separación

# ============================================
# 1. CARGAR Y PREPARAR DATOS
# ============================================

# Cargar datos
df = pd.read_csv(r'C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\fbref\jugadores_campo_2025_2026.csv')
df = df[df['minutos'] >= 400].copy()

# ============================================
# 2. FUNCIONES AUXILIARES - BÚSQUEDA DE JUGADORES
# ============================================

def normalizar_texto(texto):
    """Normaliza texto eliminando tildes y convirtiendo a minúsculas"""
    texto = str(texto).lower()
    # Eliminar tildes
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )
    return texto

def buscar_jugador(nombre_busqueda, df_disponible, max_sugerencias=5):
    """
    Busca un jugador con normalización de texto y sugiere alternativas
    Retorna: (nombre_exacto, encontrado)
    """
    jugadores_disponibles = df_disponible['jugador'].unique()
    nombre_normalizado = normalizar_texto(nombre_busqueda)
    
    # Buscar coincidencia exacta (normalizada)
    for jugador in jugadores_disponibles:
        if normalizar_texto(jugador) == nombre_normalizado:
            return jugador, True
    
    # Si no encuentra, buscar coincidencias parciales
    coincidencias = []
    for jugador in jugadores_disponibles:
        jugador_norm = normalizar_texto(jugador)
        # Coincidencia parcial (contiene el texto buscado)
        if nombre_normalizado in jugador_norm or jugador_norm in nombre_normalizado:
            coincidencias.append(jugador)
    
    if coincidencias:
        print(f"\n⚠️ '{nombre_busqueda}' no encontrado exactamente.")
        print(f"¿Quisiste decir alguno de estos?")
        for i, sugerencia in enumerate(coincidencias[:max_sugerencias], 1):
            equipo = df_disponible[df_disponible['jugador'] == sugerencia]['equipo'].iloc[0]
            print(f"  {i}. {sugerencia} ({equipo})")
        return None, False
    
    # No hay coincidencias
    print(f"\n❌ '{nombre_busqueda}' no encontrado.")
    print(f"Verifica que el jugador esté en {df_disponible['competicion'].iloc[0] if len(df_disponible) > 0 else 'la competición'}")
    print(f"y tenga al menos 300 minutos jugados.")
    return None, False

# ============================================
# 3. CONJUNTOS DE MÉTRICAS PREDEFINIDOS
# ============================================

METRICAS_HEATMAP = {
    'FW_completo': {
        'metricas': ['goles_por90', 'npxg_por90', 'xg_asistencias_por90', 
                     'tiros_a_puerta_por90', 'regates_exitosos', 'pases_progresivos'],
        'labels': ['Goles\np90', 'npxG\np90', 'xA\np90', 
                   'Tiros puerta\np90', 'Regates\nexitosos', 'Pases\nprogresivos']
    },
    'MF_completo': {
        'metricas': ['goles_asistencias_por90', 'xg_asistencias_por90', 'pases_progresivos',
                     'conducciones_progresivas', 'entradas_ganadas', 'recuperaciones'],
        'labels': ['G+A\np90', 'xA\np90', 'Pases\nprogresivos',
                   'Conducciones\nprogresivas', 'Entradas\nganadas', 'Recuperaciones']
    },
    'DF_completo': {
        'metricas': ['entradas_ganadas', 'intercepciones', 'duelos_aereos_ganados',
                     'despejes', 'porc_precision_pase', 'pases_progresivos'],
        'labels': ['Entradas\nganadas', 'Intercepciones', 'Duelos aéreos\nganados',
                   'Despejes', '% Precisión\npase', 'Pases\nprogresivos']
    }
}

# ============================================
# 3. FUNCIÓN PRINCIPAL - HEATMAP
# ============================================

def crear_heatmap_comparacion(jugadores_nombres, posicion, competicion='La Liga',
                              metricas_custom=None, labels_custom=None,
                              titulo_personalizado=None, normalizar='percentil',
                              guardar=False, filename='heatmap_comparacion.png',
                              orientacion='cuadrado'):
    """
    Crea un heatmap comparando múltiples jugadores en varias métricas
    
    Parámetros:
    - jugadores_nombres: Lista de nombres de jugadores a comparar
    - posicion: 'FW', 'MF', 'DF' (determina métricas por defecto)
    - competicion: Liga o 'Top 5 Ligas'
    - metricas_custom: Lista de métricas personalizadas (opcional, sobrescribe default)
    - labels_custom: Lista de labels personalizados (opcional)
    - titulo_personalizado: Título custom (None = auto)
    - normalizar: 'percentil' (0-100) o 'zscore' (desviaciones estándar)
    - orientacion: 'cuadrado' (1080x1080) o 'vertical' (1080x1920)
    """
    
    # Filtrar por competición
    if competicion == 'Top 5 Ligas':
        df_filtrado = df.copy()
    else:
        df_filtrado = df[df['competicion'] == competicion].copy()
    
    # Filtrar por posición
    df_posicion = df_filtrado[df_filtrado['posicion'] == posicion].copy()
    
    if len(df_posicion) == 0:
        print(f"No hay datos para posición {posicion} en {competicion}")
        return
    
    # Verificar que todos los jugadores existen usando búsqueda inteligente
    jugadores_encontrados = []
    for nombre in jugadores_nombres:
        jugador_real, encontrado = buscar_jugador(nombre, df_posicion)
        if not encontrado:
            return  # Terminar si algún jugador no se encuentra
        jugadores_encontrados.append(jugador_real)
    
    # Usar nombres encontrados (con ortografía correcta del CSV)
    jugadores_nombres = jugadores_encontrados
    
    # Determinar métricas a usar
    if metricas_custom:
        metricas = metricas_custom
        labels = labels_custom if labels_custom else [m.replace('_', '\n') for m in metricas]
    else:
        # Usar métricas predefinidas según posición
        metricas = METRICAS_HEATMAP[f'{posicion}_completo']['metricas']
        labels = METRICAS_HEATMAP[f'{posicion}_completo']['labels']
    
    # Crear matriz de datos
    n_jugadores = len(jugadores_nombres)
    n_metricas = len(metricas)
    matriz = np.zeros((n_jugadores, n_metricas))
    equipos = []
    
    for i, jugador in enumerate(jugadores_nombres):
        jugador_data = df_posicion[df_posicion['jugador'] == jugador].iloc[0]
        equipos.append(jugador_data['equipo'])
        
        for j, metrica in enumerate(metricas):
            valor = jugador_data[metrica]
            
            # Limitar porcentajes a 100
            if 'porc_' in metrica or '%' in metrica:
                valor = min(valor, 100)
            
            # Normalizar según método elegido
            if normalizar == 'percentil':
                # Calcular percentil
                percentil = (df_posicion[metrica] <= valor).sum() / len(df_posicion) * 100
                matriz[i, j] = percentil
            else:  # zscore
                # Calcular z-score y normalizar a 0-100
                mean = df_posicion[metrica].mean()
                std = df_posicion[metrica].std()
                if std > 0:
                    zscore = (valor - mean) / std
                    # Normalizar zscore a escala 0-100 (zscore típicamente entre -3 y 3)
                    matriz[i, j] = max(0, min(100, (zscore + 3) / 6 * 100))
                else:
                    matriz[i, j] = 50
    
    # Configurar tamaño de figura
    if orientacion == 'vertical':
        figsize = (10.8, 19.2)
        cell_height = 0.8
    else:  # cuadrado
        figsize = (14, 14)
        cell_height = 1.0
    
    # Crear figura
    fig, ax = plt.subplots(figsize=figsize, facecolor=COLOR_FONDO)
    ax.set_facecolor(COLOR_FONDO)
    
    # Crear colormap personalizado - PERSONALIZABLE: gradiente de colores
    colors = [COLOR_BAJO, COLOR_MEDIO_BAJO, COLOR_MEDIO, COLOR_MEDIO_ALTO, COLOR_ALTO]
    n_bins = 100
    cmap = LinearSegmentedColormap.from_list('custom', colors, N=n_bins)
    
    # Dibujar heatmap
    im = ax.imshow(matriz, cmap=cmap, aspect='auto', vmin=0, vmax=100, zorder=1)
    
    # Configurar ejes
    ax.set_xticks(np.arange(n_metricas))
    ax.set_yticks(np.arange(n_jugadores))
    
    # Labels - PERSONALIZABLE: tamaños y colores de texto
    ax.set_xticklabels(labels, fontsize=11, color=COLOR_TEXTO, weight='bold')
    ax.set_yticklabels(jugadores_nombres, fontsize=13, color=COLOR_TEXTO, weight='bold')
    
    # Rotar labels del eje X
    plt.setp(ax.get_xticklabels(), rotation=0, ha="center", va="top")
    
    # Posicionar ticks
    ax.tick_params(top=True, bottom=False, labeltop=True, labelbottom=False)
    ax.tick_params(axis='both', which='both', length=0)
    
    # Añadir nombres de equipos al lado de jugadores - PERSONALIZABLE: color de equipos
    for i, equipo in enumerate(equipos):
        ax.text(n_metricas + 0.3, i, f'({equipo})', ha='left', va='center',
               fontsize=10, color=COLOR_TEXTO_SECUNDARIO)
    
    # Añadir valores numéricos en cada celda - PERSONALIZABLE: mostrar valores
    for i in range(n_jugadores):
        for j in range(n_metricas):
            valor = matriz[i, j]
            # Color del texto según valor (claro para oscuro, oscuro para claro)
            text_color = COLOR_FONDO if valor > 50 else COLOR_TEXTO
            ax.text(j, i, f'{valor:.0f}', ha='center', va='center',
                   fontsize=10, color=text_color, weight='bold', zorder=2)
    
    # Añadir grid - PERSONALIZABLE: color y grosor del grid
    ax.set_xticks(np.arange(n_metricas + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(n_jugadores + 1) - 0.5, minor=True)
    ax.grid(which='minor', color=COLOR_GRID, linestyle='-', linewidth=2)
    
    # Título - PERSONALIZABLE: color del título
    if titulo_personalizado:
        titulo = titulo_personalizado
    else:
        titulo = f'Comparación {posicion} - {competicion}'
    
    fig.text(0.5, 0.96, titulo, ha='center', va='top',
            fontsize=26, color=COLOR_ALTO, weight='bold')
    
    # Subtítulo
    if normalizar == 'percentil':
        subtitulo = 'Valores en percentiles (0-100)'
    else:
        subtitulo = 'Valores normalizados'
    
    fig.text(0.5, 0.92, subtitulo, ha='center', va='top',
            fontsize=14, color=COLOR_TEXTO_SECUNDARIO)
    
    # Leyenda de colores - PERSONALIZABLE: posición y estilo
    cbar = plt.colorbar(im, ax=ax, orientation='horizontal', 
                       pad=0.08, shrink=0.6, aspect=30)
    cbar.ax.set_facecolor(COLOR_FONDO)
    cbar.outline.set_edgecolor(COLOR_GRID)
    cbar.outline.set_linewidth(2)
    cbar.ax.tick_params(colors=COLOR_TEXTO, labelsize=10)
    cbar.set_label('Percentil', color=COLOR_TEXTO, fontsize=11, weight='bold')
    
    # Nota al pie - PERSONALIZABLE: color de la nota
    fig.text(0.5, 0.02, f'Min. 400 min jugados | {len(df_posicion)} jugadores analizados | Datos: FBref (Opta) | @manuelperezruda',
            ha='center', va='center', fontsize=9, color=COLOR_TEXTO_SECUNDARIO)
    
    plt.tight_layout(rect=[0, 0.03, 1, 0.90])
    
    if guardar:
        plt.savefig(filename, dpi=300, facecolor=COLOR_FONDO, bbox_inches='tight')
        print(f"Gráfico guardado como '{filename}'")
    
    plt.show()
    
    # Mostrar resumen en consola
    print(f"\nHeatmap de comparación - {posicion} en {competicion}:")
    print("="*70)
    for i, jugador in enumerate(jugadores_nombres):
        print(f"\n{jugador} ({equipos[i]}):")
        for j, (metrica, label) in enumerate(zip(metricas, labels)):
            print(f"  {label.replace(chr(10), ' '):25s}: {matriz[i, j]:5.1f}º percentil")


# ============================================
# 4. EJEMPLOS DE USO
# ============================================

print("\n" + "="*70)
print("HEATMAP DE COMPARACIÓN MULTI-JUGADOR")
print("="*70)

print("\n📊 EJEMPLOS DE USO:")

print("\n1️⃣ Heatmap básico (métricas predefinidas):")
print("   crear_heatmap_comparacion(['Pedri', 'Gavi', 'Bellingham'], 'MF')")

print("\n2️⃣ Comparar delanteros:")
print("   crear_heatmap_comparacion(['Lewandowski', 'Mbappé', 'Haaland'], 'FW',")
print("                             competicion='Top 5 Ligas')")

print("\n3️⃣ Con métricas personalizadas:")
print("   metricas = ['goles_por90', 'asistencias_por90', 'tiros_por90', 'regates_exitosos']")
print("   labels = ['Goles', 'Asistencias', 'Tiros', 'Regates']")
print("   crear_heatmap_comparacion(['Yamal', 'Vinicius'], 'FW',")
print("                             metricas_custom=metricas, labels_custom=labels)")

print("\n4️⃣ Con título personalizado:")
print("   crear_heatmap_comparacion(['Pedri', 'Gavi'], 'MF',")
print("                             titulo_personalizado='Duelo en el Camp Nou')")

print("\n5️⃣ Formato vertical (stories):")
print("   crear_heatmap_comparacion(['Pedri', 'Gavi', 'De Jong'], 'MF', orientacion='vertical')")

print("\n💡 CONSEJO: Compara entre 2 y 6 jugadores para mejor visualización")

In [ ]:
crear_heatmap_comparacion(['Julian Alvarez', 'Kylian Mbappé', 'Erling Haaland'], 'FW',
                             competicion='Top 5 Ligas')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from mplsoccer import Radar
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================
# CONFIGURACIÓN DE COLORES - PERSONALIZABLE
# ============================================
COLOR_PRINCIPAL = '#00f2c1'         # Color jugador principal (cyan)
COLOR_SECUNDARIO = '#ff6b35'        # Color promedio/comparación (naranja)
COLOR_SIMILAR = '#a0e8af'           # Color jugadores similares (verde claro)
COLOR_FONDO = '#24282a'             # Fondo general
COLOR_TEXTO = '#ffffff'             # Texto principal
COLOR_TEXTO_SECUNDARIO = '#808080'  # Texto secundario

RUTA_BASE = r'C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\assets\viz'

# ============================================
# CARGAR DATOS
# ============================================
df = pd.read_csv(r'C:\Users\manue\OneDrive\Escritorio\Proyecto WhoScored\data\raw\fbref\jugadores_campo_2025_2026.csv')
df = df[df['minutos'] >= 300].copy()

# ============================================
# MÉTRICAS POR POSICIÓN
# ============================================
METRICAS_RADAR = {
    'FW': {
        'metricas': ['goles_por90', 'npxg_por90', 'tiros_a_puerta_por90', 'xg_asistencias_por90',
                     'pases_progresivos', 'conducciones_progresivas', 'regates_exitosos', 
                     'toques_area_rival', 'pases_tercio_final'],
        'labels': ['Goles\np90', 'npxG\np90', 'Tiros puerta\np90', 'xA p90',
                   'Pases\nprog.', 'Conduc.\nprog.', 'Regates', 'Toques\nÁrea', 'Pases\ntercio final']
    },
    'MF': {
        'metricas': ['xg_asistencias_por90', 'pases_progresivos', 'conducciones_progresivas',
                     'porc_precision_pase', 'pases_tercio_final', 'goles_asistencias_por90',
                     'entradas_ganadas', 'intercepciones', 'recuperaciones'],
        'labels': ['xA p90', 'Pases\nprog.', 'Conduc.\nprog.', '% Precisión\npase',
                   'Pases\ntercio final', 'G+A p90', 'Entradas', 'Intercep.', 'Recup.']
    },
    'DF': {
        'metricas': ['entradas_ganadas', 'intercepciones', 'duelos_aereos_ganados',
                     'porc_duelos_aereos_ganados', 'despejes', 'bloqueos', 
                     'recuperaciones', 'porc_precision_pase', 'pases_progresivos'],
        'labels': ['Entradas', 'Intercep.', 'Duelos\naéreos', '% Duelos\naéreos',
                   'Despejes', 'Bloqueos', 'Recup.', '% Precisión\npase', 'Pases\nprog.']
    }
}

METRICAS_SIMILITUD = {
    'FW': ['goles_por90', 'npxg_por90', 'xg_asistencias_por90', 'tiros_por90', 
           'regates_exitosos', 'pases_progresivos'],
    'MF': ['goles_asistencias_por90', 'xg_asistencias_por90', 'pases_progresivos',
           'conducciones_progresivas', 'entradas_ganadas', 'recuperaciones'],
    'DF': ['entradas_ganadas', 'intercepciones', 'duelos_aereos_ganados',
           'despejes', 'porc_precision_pase', 'pases_progresivos']
}

def encontrar_jugadores_similares(jugador_nombre, posicion, competicion, df_data, n=4):
    """Encuentra los N jugadores más similares usando similitud del coseno"""
    if competicion == 'Top 5 Ligas':
        df_filtrado = df_data.copy()
    else:
        df_filtrado = df_data[df_data['competicion'] == competicion].copy()
    
    df_posicion = df_filtrado[df_filtrado['posicion'] == posicion].copy()
    metricas = METRICAS_SIMILITUD[posicion]
    X = df_posicion[metricas].fillna(0).values
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    idx_jugador = df_posicion[df_posicion['jugador'] == jugador_nombre].index[0]
    idx_en_matriz = df_posicion.index.get_loc(idx_jugador)
    vector_jugador = X_scaled[idx_en_matriz].reshape(1, -1)
    similitudes = cosine_similarity(vector_jugador, X_scaled)[0]
    indices_similares = np.argsort(similitudes)[::-1][1:n+1]
    jugadores_similares = df_posicion.iloc[indices_similares]['jugador'].tolist()
    return jugadores_similares

# ============================================
# INFOGRAFÍA INDIVIDUAL
# ============================================
def crear_infografia_individual(jugador_nombre, posicion, competicion='La Liga',
                                metrica_scatter_x='pases_progresivos',
                                metrica_scatter_y='xg_asistencias_por90',
                                metrica_ranking='recuperaciones',
                                top_ranking=10,
                                mostrar_posicion=True,
                                guardar=True):
    """
    Infografía individual completa
    
    Parámetros:
    - mostrar_posicion: Si True, muestra la posición en el header. Si False, la oculta.
    """
    
    # Filtrar datos
    if competicion == 'Top 5 Ligas':
        df_filtrado = df.copy()
    else:
        df_filtrado = df[df['competicion'] == competicion].copy()
    
    df_posicion = df_filtrado[df_filtrado['posicion'] == posicion].copy()
    jugador_data = df_posicion[df_posicion['jugador'] == jugador_nombre].iloc[0]
    jugadores_similares = encontrar_jugadores_similares(jugador_nombre, posicion, competicion, df, n=4)
    
    # ============================================
    # CONFIGURACIÓN DE LAYOUT - PERSONALIZABLE
    # ============================================
    # Ajustar height_ratios para cambiar tamaño de secciones
    # [header, datos_basicos, fila_radar/heatmap, fila_scatter/ranking, pie]
    fig = plt.figure(figsize=(14, 20), facecolor=COLOR_FONDO)
    gs = GridSpec(4, 2, figure=fig, height_ratios=[0.5, 6, 6.2, 5],
                  hspace=0.20,  # Espacio vertical entre filas (reducido)
                  wspace=0.25,  # Espacio horizontal entre columnas
                  left=0.03,    # ⬅️ Mover TODO hacia la IZQUIERDA (aumentar)
                  right=0.91,   # ➡️ Mover TODO hacia la DERECHA (disminuir)
                  top=0.88,     # ⬆️ SUBIR TODO (aumentar = sube)
                  bottom=0.03)  # ⬇️ BAJAR TODO (aumentar = baja)
    
    # ============================================
    # 1. HEADER - PERSONALIZABLE (posición Y del texto)
    # ============================================
    # Nombre del jugador
    fig.text(0.5, 0.985,  # (X, Y) - Y más alto = más arriba
            jugador_nombre, ha='center', va='top',
            fontsize=34, color=COLOR_PRINCIPAL, weight='bold')
    
    # Equipo y competición (con/sin posición)
    if mostrar_posicion:
        header_text = f"{jugador_data['equipo']} | {posicion} | {competicion}"
    else:
        header_text = f"{jugador_data['equipo']} | {competicion}"
    
    fig.text(0.5, 0.965, header_text, ha='center', va='top',
            fontsize=18, color=COLOR_TEXTO)
    
    # Datos básicos
    partidos = int(jugador_data['partidos_jugados'])
    minutos = int(jugador_data['minutos'])
    goles = int(jugador_data['goles'])
    asistencias = int(jugador_data['asistencias'])
    datos_basicos = f"PJ: {partidos}  |  MIN: {minutos}  |  G: {goles}  |  A: {asistencias}"
    
    fig.text(0.5, 0.945, datos_basicos, ha='center', va='top',
            fontsize=14, color=COLOR_TEXTO_SECUNDARIO)
    
    # ============================================
    # 2. RADAR CHART - MÁS GRANDE CON LEYENDA
    # ============================================
    # PERSONALIZABLE: Cambiar gs[1, 0] a gs[fila, columna] para mover
    ax_radar = fig.add_subplot(gs[1, 0])
    # SUBIR EL RADAR (mueve su bbox unos puntos hacia arriba)
    pos = ax_radar.get_position()
    ax_radar.set_position([pos.x0, pos.y0 + 0.02, pos.width, pos.height])
    # Mover ligeramente a la izquierda
    pos_r = ax_radar.get_position()
    ax_radar.set_position([pos_r.x0 - 0.02, pos_r.y0, pos_r.width, pos_r.height])
    
    metricas = METRICAS_RADAR[posicion]['metricas']
    labels = METRICAS_RADAR[posicion]['labels']
    valores_jugador = []
    valores_promedio = []
    
    for metrica in metricas:
        val_jug = jugador_data[metrica]
        val_prom = df_posicion[metrica].mean()
        if 'porc_' in metrica:
            val_jug = min(val_jug, 100)
            val_prom = min(val_prom, 100)
        valores_jugador.append(val_jug)
        valores_promedio.append(val_prom)
    
    low = [0] * len(labels)
    high = []
    for i, (vj, vp, label) in enumerate(zip(valores_jugador, valores_promedio, labels)):
        if '%' in label or 'porc_' in metricas[i]:
            high.append(100)
        else:
            high.append(max(vj, vp) * 1.3)
    
    # PERSONALIZABLE: num_rings, ring_width para cambiar anillos
    radar = Radar(labels, low, high, num_rings=4, ring_width=1, center_circle_radius=1)
    radar.setup_axis(ax=ax_radar, facecolor=COLOR_FONDO)
    ax_radar.set_facecolor(COLOR_FONDO)
    
    # PERSONALIZABLE: facecolor y edgecolor de los anillos
    radar.draw_circles(ax=ax_radar, facecolor='#3a3f42', edgecolor='#808080', linewidth=1.5)
    
    # PERSONALIZABLE: alpha (transparencia), linewidth (grosor líneas)
    radar_output = radar.draw_radar_compare(valores_promedio, valores_jugador, ax=ax_radar,
                                            kwargs_radar={'facecolor': COLOR_SECUNDARIO, 'alpha': 0.4,
                                                        'edgecolor': COLOR_SECUNDARIO, 'linewidth': 2.5},
                                            kwargs_compare={'facecolor': COLOR_PRINCIPAL, 'alpha': 0.55,
                                                          'edgecolor': COLOR_PRINCIPAL, 'linewidth': 2.5})
    
    # PERSONALIZABLE: fontsize de labels
    radar.draw_range_labels(ax=ax_radar, fontsize=11, color=COLOR_TEXTO)
    radar.draw_param_labels(ax=ax_radar, fontsize=12, color=COLOR_TEXTO)
    
    # Leyenda del radar
    legend_elements = [
        Patch(facecolor=COLOR_PRINCIPAL, alpha=0.6, label=jugador_nombre),
        Patch(facecolor=COLOR_SECUNDARIO, alpha=0.4, label='Promedio Liga')
    ]
    leg = ax_radar.legend(handles=legend_elements, loc='upper right', fontsize=10,
                   frameon=False, bbox_to_anchor=(1.1, 1))
    
    # Colorear el texto de la leyenda
    for t in leg.get_texts():
        t.set_color(COLOR_TEXTO)
    
    # PERSONALIZABLE: fontsize, pad (separación del título)
    ax_radar.set_title('Perfil Técnico vs Promedio Liga', fontsize=15, 
                      color=COLOR_TEXTO, weight='bold', pad=25)
    
    # ============================================
    # 3. HEATMAP - MÁS GRANDE, COLUMNAS EN DIAGONAL
    # ============================================
    # PERSONALIZABLE: gs[1, 1] = fila 1, columna 1
    ax_heatmap = fig.add_subplot(gs[1, 1])
    # MOVER A LA IZQUIERDA LIGERAMENTE
    pos_h = ax_heatmap.get_position()
    ax_heatmap.set_position([pos_h.x0 - 0.01, pos_h.y0, pos_h.width, pos_h.height])
    
    todos_jugadores = [jugador_nombre] + jugadores_similares
    metricas_heat = metricas[:6]
    labels_heat = METRICAS_RADAR[posicion]['labels'][:6]
    
    matriz_real = np.zeros((5, 6))
    for i, jug in enumerate(todos_jugadores):
        jug_data = df_posicion[df_posicion['jugador'] == jug].iloc[0]
        for j, met in enumerate(metricas_heat):
            matriz_real[i, j] = jug_data[met]
    
    matriz_color = np.zeros((5, 6))
    for j in range(matriz_real.shape[1]):
        col_min = matriz_real[:, j].min()
        col_max = matriz_real[:, j].max()
        if col_max > col_min:
            matriz_color[:, j] = ((matriz_real[:, j] - col_min) / (col_max - col_min)) * 100
        else:
            matriz_color[:, j] = 50
    
    # PERSONALIZABLE: colores del gradiente del heatmap
    colors = ['#1a1d1f', '#ff6b35', '#ffb547', '#a0e8af', '#00f2c1']
    cmap = LinearSegmentedColormap.from_list('custom', colors, N=100)
    
    im = ax_heatmap.imshow(matriz_color, cmap=cmap, aspect='auto', vmin=0, vmax=100)
    
    ax_heatmap.set_xticks(np.arange(6))
    ax_heatmap.set_yticks(np.arange(5))
    
    # COLUMNAS EN DIAGONAL (rotation=45)
    # PERSONALIZABLE: rotation (ángulo), fontsize, ha (alineación)
    ax_heatmap.set_xticklabels(labels_heat, fontsize=11, color=COLOR_TEXTO, 
                              rotation=45, ha='left')
    ax_heatmap.set_yticklabels(todos_jugadores, fontsize=11, color=COLOR_TEXTO)
    
    # PERSONALIZABLE: Separación del título (pad en ax_heatmap.xaxis)
    ax_heatmap.tick_params(top=True, bottom=False, labeltop=True, labelbottom=False, 
                          length=0, pad=6)
    
    # Destacar jugador principal
    ax_heatmap.get_yticklabels()[0].set_color(COLOR_PRINCIPAL)
    ax_heatmap.get_yticklabels()[0].set_weight('bold')
    ax_heatmap.get_yticklabels()[0].set_fontsize(12)
    
    # Valores en celdas
    for i in range(5):
        for j in range(6):
            valor_real = matriz_real[i, j]
            valor_color = matriz_color[i, j]
            text_color = COLOR_FONDO if valor_color > 50 else COLOR_TEXTO
            
            if valor_real == int(valor_real):
                val_display = str(int(valor_real))
            else:
                val_display = f'{valor_real:.1f}'
            
            # PERSONALIZABLE: fontsize de valores en celdas
            ax_heatmap.text(j, i, val_display, ha='center', va='center',
                          fontsize=10, color=text_color, weight='bold')
    
    # Grid
    ax_heatmap.set_xticks(np.arange(6 + 1) - 0.5, minor=True)
    ax_heatmap.set_yticks(np.arange(5 + 1) - 0.5, minor=True)
    ax_heatmap.grid(which='minor', color='#2e3336', linestyle='-', linewidth=2)
    
    # PERSONALIZABLE: pad (separación del título)
    ax_heatmap.set_title('Comparación con Jugadores Similares', fontsize=15,
                        color=COLOR_TEXTO, weight='bold', pad=25)
    
    # ============================================
    # 4. SCATTER PLOT
    # ============================================
    # PERSONALIZABLE: gs[2, 0] para mover
    ax_scatter = fig.add_subplot(gs[2, 0])
    ax_scatter.set_facecolor(COLOR_FONDO)
    
    x_data = df_posicion[metrica_scatter_x].values
    y_data = df_posicion[metrica_scatter_y].values
    nombres = df_posicion['jugador'].values
    
    mask_similares = np.isin(nombres, todos_jugadores)
    
    # PERSONALIZABLE: s (tamaño puntos), alpha (transparencia), color
    ax_scatter.scatter(x_data[~mask_similares], y_data[~mask_similares],
                      s=80, color='#404040', alpha=0.3, zorder=1)
    
    mask_sim_solo = np.isin(nombres, jugadores_similares)
    ax_scatter.scatter(x_data[mask_sim_solo], y_data[mask_sim_solo],
                      s=180, color=COLOR_SIMILAR, alpha=0.8, zorder=2, 
                      edgecolors=COLOR_TEXTO, linewidth=1.5)
    
    mask_principal = nombres == jugador_nombre
    ax_scatter.scatter(x_data[mask_principal], y_data[mask_principal],
                      s=280, color=COLOR_PRINCIPAL, alpha=0.9, zorder=3,
                      edgecolors=COLOR_TEXTO, linewidth=2.5)
    
    # Promedios
    x_prom = np.mean(x_data)
    y_prom = np.mean(y_data)
    
    # PERSONALIZABLE: linewidth, alpha de líneas promedio
    ax_scatter.axvline(x_prom, color= COLOR_SECUNDARIO, linestyle='--', linewidth=1.5, alpha=0.5, zorder=0)
    ax_scatter.axhline(y_prom, color=COLOR_SECUNDARIO, linestyle='--', linewidth=1.5, alpha=0.5, zorder=0)
    
    ax_scatter.text(x_prom, ax_scatter.get_ylim()[1], f'{x_prom:.1f}',
                   ha='center', va='bottom', fontsize=10, color=COLOR_TEXTO,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor=COLOR_FONDO, edgecolor=COLOR_TEXTO, alpha=0.8))
    ax_scatter.text(ax_scatter.get_xlim()[1], y_prom, f'{y_prom:.1f}',
                   ha='left', va='center', fontsize=10, color=COLOR_TEXTO,
                   bbox=dict(boxstyle='round,pad=0.3', facecolor=COLOR_FONDO, edgecolor=COLOR_TEXTO, alpha=0.8))
    
    # Nombres
    for nombre in todos_jugadores:
        mask = nombres == nombre
        if mask.any():
            x_pos = x_data[mask][0]
            y_pos = y_data[mask][0]
            
            if nombre == jugador_nombre:
                color_nombre = COLOR_PRINCIPAL
                size = 11
                weight = 'bold'
            else:
                color_nombre = COLOR_SIMILAR
                size = 9
                weight = 'normal'
            
            # PERSONALIZABLE: fontsize de nombres
            ax_scatter.text(x_pos, y_pos + (ax_scatter.get_ylim()[1] - ax_scatter.get_ylim()[0]) * 0.03,
                          nombre, ha='center', va='bottom', fontsize=size,
                          color=color_nombre, weight=weight,
                          bbox=dict(boxstyle='round,pad=0.2', facecolor=COLOR_FONDO, alpha=0.7, edgecolor='none'))
    
    # PERSONALIZABLE: fontsize de ejes
    ax_scatter.set_xlabel(metrica_scatter_x.replace('_', ' ').title(), 
                         fontsize=13, color=COLOR_TEXTO, weight='bold')
    ax_scatter.set_ylabel(metrica_scatter_y.replace('_', ' ').title(),
                         fontsize=13, color=COLOR_TEXTO, weight='bold')
    ax_scatter.tick_params(colors=COLOR_TEXTO, labelsize=11)
    ax_scatter.grid(alpha=0.2, color=COLOR_TEXTO_SECUNDARIO)
    
    for spine in ax_scatter.spines.values():
        spine.set_edgecolor(COLOR_TEXTO_SECUNDARIO)
    
    ax_scatter.set_title('Posicionamiento en la Liga', fontsize=15,
                        color=COLOR_TEXTO, weight='bold', pad=18)
    
    # ============================================
    # 5. RANKING - MÁS GRANDE CON EQUIPOS
    # ============================================
    # PERSONALIZABLE: gs[2, 1] para mover
    ax_ranking = fig.add_subplot(gs[2, 1])
    ax_ranking.set_facecolor(COLOR_FONDO)
    ax_ranking.axis('off')
    ax_ranking.set_xlim(0, 10)
    ax_ranking.set_ylim(0, top_ranking + 2.5)
    # Estira su bbox un poco a la izquierda y a la derecha/alto
    pos = ax_ranking.get_position()
    ax_ranking.set_position([pos.x0 - 0.015, pos.y0 - 0.005,  # mover origen
                            pos.width + 0.06, pos.height + 0.04])  # crecer ancho/alto
    # Mover hacia abajo ligeramente
    pos_r = ax_ranking.get_position()
    ax_ranking.set_position([pos_r.x0, pos_r.y0 - 0.01, pos_r.width, pos_r.height])
    
    df_top = df_posicion.nlargest(top_ranking, metrica_ranking)
    rankings = list(range(1, len(df_top) + 1))
    jugadores_rank = df_top['jugador'].values
    valores_rank = df_top[metrica_ranking].values
    max_valor = valores_rank.max()
    
    # PERSONALIZABLE: fontsize del título
    ax_ranking.text(5, top_ranking + 2, f'Top {top_ranking} - {metrica_ranking.replace("_", " ").title()}',
                   ha='center', va='center', fontsize=15, color=COLOR_TEXTO, weight='bold')
    
    # Filas
    for i, (rank, jug, valor) in enumerate(zip(rankings, jugadores_rank, valores_rank)):
        y_pos = top_ranking - i + 0.5
        
        if jug == jugador_nombre:
            color = COLOR_PRINCIPAL
            weight = 'bold'
            fontsize_rank = 13
            fontsize_nombre = 12
        elif jug in jugadores_similares:
            color = COLOR_SIMILAR
            weight = 'bold'
            fontsize_rank = 12
            fontsize_nombre = 12
        else:
            color = COLOR_TEXTO
            weight = 'normal'
            fontsize_rank = 12
            fontsize_nombre = 12
        
        # PERSONALIZABLE: posiciones X de ranking, nombre, equipo
        ax_ranking.text(0.5, y_pos, str(rank), ha='center', va='center',
                       fontsize=fontsize_rank, color=color, weight=weight)
        
        ax_ranking.text(1.10, y_pos, jug, ha='left', va='center',
                       fontsize=fontsize_nombre, color=color, weight=weight)
        
        equipo = df_posicion[df_posicion['jugador'] == jug]['equipo'].values[0]
        ax_ranking.text(3.60, y_pos, f'({equipo})', ha='left', va='center',
                       fontsize=9, color=COLOR_TEXTO_SECUNDARIO)
        
        # PERSONALIZABLE: posición X y ancho de barras
        bar_width = (valor / max_valor) * 3.5
        ax_ranking.add_patch(plt.Rectangle((5.35, y_pos - 0.35), bar_width, 0.7,
                                          facecolor=color, alpha=0.6))
        
        # Valor
        if valor == int(valor):
            val_display = str(int(valor))
        else:
            val_display = f'{valor:.2f}'
        
        ax_ranking.text(9.5, y_pos, val_display, ha='right', va='center',
                       fontsize=11, color=color, weight='bold')
    
    # ============================================
    # NOTA AL PIE - PERSONALIZABLE (posición Y)
    # ============================================
    fig.text(0.5, 0.01, f'Min. 400 min | Datos: FBref (Opta) | @manuelperezruda',
            ha='center', va='center', fontsize=10, color=COLOR_TEXTO_SECUNDARIO)
    
    # ============================================
    # GUARDAR
    # ============================================
    if guardar:
        carpeta_jugador = os.path.join(RUTA_BASE, jugador_nombre.replace(' ', '_'))
        os.makedirs(carpeta_jugador, exist_ok=True)
        filename = os.path.join(carpeta_jugador, f'{jugador_nombre.replace(" ", "_")}_infografia.png')
        plt.savefig(filename, dpi=300, facecolor=COLOR_FONDO, bbox_inches='tight')
        print(f"✅ Infografía guardada: {filename}")
    
    plt.show()


# ============================================
# EJEMPLOS
# ============================================
print("\n" + "="*70)
print("INFOGRAFÍA INDIVIDUAL - VERSIÓN FINAL")
print("="*70)
print("\nEjemplo:")
print("crear_infografia_individual('Pedri', 'MF',")
print("                            competicion='La Liga',")
print("                            metrica_scatter_x='pases_progresivos',")
print("                            metrica_scatter_y='xg_asistencias_por90',")
print("                            metrica_ranking='recuperaciones',")
print("                            mostrar_posicion=True,")
print("                            guardar=True)")

In [ ]:
crear_infografia_individual('Pedri', 'MF', competicion='Top 5 Ligas',
                            metrica_scatter_x='pases_progresivos',
                            metrica_scatter_y='pases_tercio_final',
                            metrica_ranking='recuperaciones',
                            top_ranking=10, guardar=True)